# Evaluación Espacial de Conservación (EOO, AOO), Biodiversidad Local y Estructura Genética en Flora Nativa de Chile

**Asignatura:** Biodiversidad y Conservación Biológica (2026)  
**Institución:** Facultad de Ciencias Forestales, Universidad de Concepción (Chile)  
**Docente titular:** Dr. Rodrigo Hasbún Zaror | **Ayudantía y desarrollo:** Mauricio Aguirre-Díaz ([`magdzraw/byc-2026`](https://github.com/magdzraw/byc-2026))  
**Especies configuradas:** Alerce (*Fitzroya cupressoides*), Araucaria (*Araucaria araucana*), Lleuque (*Prumnopitys andina*), Queule (*Gomortega keule*) y Ruil (*Nothofagus alessandrii*).

---

## Marco Normativo y Metodológico

Este cuaderno implementa un flujo reproducible en Python para estimar parámetros geográficos del **Criterio B** de amenaza de extinción e índices locales de diversidad de flora vascular acompañante, alineado con:

1. **Reglamento de Clasificación de Especies Silvestres de Chile (RCE — D.S. N° 29/2011 MMA):** Adopta las categorías y criterios de la UICN y las *Directrices para Emplear los Criterios de la Lista Roja de la UICN a Nivel Regional y Nacional (v4.0)*. En el modo por defecto (`RCE_CHILE`), la evaluación se restringe a registros en **territorio chileno (`CL`)** recortados por el límite nacional de alta resolución (Paso 1 regional). Para especies compartidas con Argentina (*Alerce*, *Araucaria*, *Lleuque*), la evaluación formal requiere considerar en un Paso 2 el eventual *efecto de rescate* desde poblaciones vecinas.
2. **Directrices de la Lista Roja Global de la UICN (v16, 2024):** Disponible seleccionando `ESCALA_EVALUACION = 'UICN_GLOBAL'` (incorpora registros de Chile y Argentina, `CL` + `AR`, y calcula el Polígono Convexo Mínimo global sin exclusión de disyunciones).
3. **Planes RECOGE (MMA-FAO-GEF) y literatura nacional:** Para especies con distribución sumamente restringida y fragmentada en rodales pequeños (como *Gomortega keule* en el **Plan RECOGE del Queule, Mayo 2022, pp. 6–7**, que reporta AOO en grilla de $1 \times 1\text{ km}$, o *Nothofagus alessandrii* en el **Plan RECOGE del Ruil, D.S. N° 42/2020**), el cuaderno calcula **dos grillas de AOO claramente diferenciadas**:
   * **AOO Oficial UICN / RCE ($2 \times 2\text{ km} = 4\text{ km}^2$):** Escala obligatoria para contrastar con los umbrales del Criterio B2 (UICN Sec. 4.10), optimizada por desplazamiento de origen para obtener el mínimo número de celdas ocupadas.
   * **AOO de Escala Fina / Literatura ($1 \times 1\text{ km} = 1\text{ km}^2$):** Capa complementaria de micro-ocupación comparable con los Planes RECOGE (no se compara con umbrales B2 para evitar sesgo de escala).

### Fuentes de Datos Utilizadas
| Recurso | Origen / API | Uso en el análisis |
| :--- | :--- | :--- |
| **Localidades de referencia** | `Coordenadas_codigo.csv` (delimitado por `;`) | Sitios de muestreo genético/terreno (con `Codigo`), validación y centros de la malla hexagonal de biodiversidad. |
| **Ocurrencias de la especie focal** | **GBIF API v1** (`/v1/species/match` + `/v1/occurrence/search`) | Cálculo de EOO y AOO tras curaduría taxonómica, espacial y biogeográfica. |
| **Flora vascular acompañante** | **GBIF API v1** (`phylumKey = 7707728`, *Tracheophyta*) | Cálculo de riqueza ($S$), Shannon ($H'$), Pielou ($J'$), Gini-Simpson ($1-D$), Chao1 y Cobertura ($\hat{C}$). |
| **Límites nacionales y línea de costa** | **GADM v4.1** (alta resolución; respaldo: **Natural Earth 1:10m**) | Máscara terrestre precisa (incluyendo archipiélagos, fiordos, Chiloé y regiones) para recorte RCE Chile / Global. |
| **Estructura genética (Módulo 11 opcional)** | Resultados de **STRUCTURE** (`.zip` de `Results/`, archivo `*_f` o `.csv`) | Cálculo de Evanno ($\Delta K$), consenso entre réplicas tipo CLUMPP, figura compuesta A+B+C, tortas SVG en Folium, IDW opcional y capas `.gpkg` para QGIS. |

## 1. Instalación e Importación de Librerías

Instalamos las dependencias espaciales, estadísticas y de visualización requeridas en Google Colab (`geopandas`, `shapely`, `pyogrio`, `folium`, `mapclassify`, `matplotlib`, `scipy`, `requests`).

In [ ]:
# Instalación de librerías espaciales y científicas (en Colab; si ya están instaladas, pip las omite)
!pip install geopandas requests folium pyogrio matplotlib mapclassify shapely scipy --quiet

import os
import re
import io
import math
import base64
import zipfile
import warnings
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import Polygon, Point, box
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patheffects import withStroke
import folium

# Sesión HTTP con reintentos automáticos ante cortes transitorios de red/SSL (GBIF / GADM)
SESSION_HTTP = requests.Session()
_retry_cfg = Retry(total=4, backoff_factor=0.6, status_forcelist=[429, 500, 502, 503, 504])
SESSION_HTTP.mount("https://", HTTPAdapter(max_retries=_retry_cfg))
SESSION_HTTP.mount("http://", HTTPAdapter(max_retries=_retry_cfg))

warnings.filterwarnings("ignore", category=UserWarning, module="pyogrio")
print("Librerías cargadas correctamente.")

## 2. Panel de Configuración del Estudio

Selecciona la **especie** y la **escala de evaluación**:
* **`RCE_CHILE` (por defecto):** Utiliza exclusivamente ocurrencias en **Chile (`CL`)** y recorta el EOO a la superficie terrestre chilena de alta resolución (GADM 4.1), siguiendo el Paso 1 del RCE-MMA.
* **`UICN_GLOBAL`:** Incluye ocurrencias nativas de **Chile y Argentina (`CL`, `AR`)** para evaluar el rango global de especies transandinas (*Alerce*, *Araucaria*, *Lleuque*).

> **Nota sobre la escala de AOO y literatura chilena:**  
> * **Estándar UICN / RCE (Criterio B2):** Siempre **$2000\text{ m}$ ($2 \times 2\text{ km} = 4\text{ km}^2$)** para todas las especies (UICN v16, Sec. 4.10).  
> * **Escala fina de literatura (complementaria):** En *Queule* ($1000\text{ m}$, Plan RECOGE Queule MMA-FAO-GEF 2022, pp. 6–7: EOO $4.392{,}5\text{ km}^2$ y AOO $86\text{ km}^2$ en grilla de $1\text{ km}^2$) y *Ruil* ($1000\text{ m}$, fragmentos de 1–60 ha en Cordillera de la Costa del Maule, Plan RECOGE Ruil D.S. 42/2020) se genera una segunda grilla fina para análisis ecológico comparativo.

In [ ]:
#@title Parámetros principales del análisis { display-mode: "form" }
ESPECIE_SELECCIONADA = "Lleuque"  #@param ["Alerce", "Araucaria", "Lleuque", "Queule", "Ruil"]
ESCALA_EVALUACION = "RCE_CHILE"  #@param ["RCE_CHILE", "UICN_GLOBAL"]
CSV_UNIFICADO = "Coordenadas_codigo.csv"  #@param {type:"string"}
INCLUIR_SITIOS_CSV_EN_UICN = True  #@param {type:"boolean"}
INCERTIDUMBRE_MAXIMA_M = 2000  #@param {type:"integer"}
PASOS_OPTIMIZACION_AOO = 10  #@param {type:"integer"}
RADIO_HEXAGONO_M = 2500  #@param {type:"integer"}
BUFFER_LOCAL_M = 7500  #@param {type:"integer"}
N_MIN_ESFUERZO = 15  #@param {type:"integer"}
MAX_REGISTROS_FLORA_PARCHE = 6000  #@param {type:"integer"}

# Países consultados en GBIF según escala normativa
PAISES_CONSULTA = ["CL"] if ESCALA_EVALUACION == "RCE_CHILE" else ["CL", "AR"]

# Taxonomía canónica (GBIF Backbone resuelve automáticamente sinónimos mediante usageKey)
TAXONOMIA_ESPECIES = {
    "Alerce":    "Fitzroya cupressoides",
    "Araucaria": "Araucaria araucana",
    "Lleuque":   "Prumnopitys andina",
    "Queule":    "Gomortega keule",
    "Ruil":      "Nothofagus alessandrii",
}

# Grilla oficial UICN B2 (obligatoria: 2x2 km = 4 km2)
GRID_AOO_UICN_M = 2000
AREA_CELDA_UICN_KM2 = (GRID_AOO_UICN_M / 1000.0) ** 2

# Grilla fina complementaria según literatura chilena / Planes RECOGE (None si literatura usa 2x2 km)
AOO_LITERATURA_CONFIG = {
    "Queule": {
        "grid_m": 1000,
        "referencia": "Plan RECOGE Queule (MMA-FAO-GEF, 2022, pp. 6-7; LEP-UdeC, 2019): EOO = 4.392,5 km², AOO (1x1 km) = 86 km²; RCE = EN (D.S. 151/2007)."
    },
    "Ruil": {
        "grid_m": 1000,
        "referencia": "Plan RECOGE Ruil (D.S. 42/2020 MMA; Bustamante & Castor, 1998): distribución restringida en Maule (~352 ha en >180 fragmentos de 1-60 ha); RCE = EN."
    },
    "Lleuque": {
        "grid_m": None,
        "referencia": "Fichas RCE-MMA / UICN: EOO nacional reportado ~36.962 km², AOO estándar en 2x2 km; RCE = VU."
    },
    "Araucaria": {
        "grid_m": None,
        "referencia": "RCE-MMA / CONAF / UICN: poblaciones disyuntas Costa (Nahuelbuta) y Andes; AOO estándar 2x2 km; RCE/UICN = EN (por Criterio A / subpoblaciones)."
    },
    "Alerce": {
        "grid_m": None,
        "referencia": "RCE-MMA / Lara et al. / UICN: poblaciones en Costa, Depresión Intermedia (ej. Lahuen Ñadi) y Andes; AOO estándar 2x2 km; RCE/UICN = EN (por Criterio A)."
    },
}

# Envolventes biogeográficas nativas para depurar registros cultivados/urbanos/erróneos en GBIF
# (calibradas con distribución documentada en literatura y Planes RECOGE)
ENVOLVENTES_SUBPOBLACIONES = {
    "Alerce": {
        "Costa / Cordillera Pelada - Chiloé": {
            "lat_min": -43.20, "lat_max": -39.80,
            "lon_min": -74.35, "lon_max": -73.15
        },
        "Depresión Intermedia (Llanquihue / Lahuen Ñadi)": {
            "lat_min": -41.75, "lat_max": -41.00,
            "lon_min": -73.15, "lon_max": -72.75
        },
        "Andes / Los Lagos - Palena (- Chubut)": {
            "lat_min": -43.60, "lat_max": -40.65,
            "lon_min": -72.75, "lon_max": -71.15
        }
    },
    "Araucaria": {
        "Costa / Cordillera de Nahuelbuta": {
            "lat_min": -38.65, "lat_max": -37.50,
            "lon_min": -73.45, "lon_max": -72.85
        },
        "Andes / Biobío - Los Ríos (- Neuquén)": {
            "lat_min": -40.35, "lat_max": -37.25,
            "lon_min": -72.15, "lon_max": -70.50
        }
    },
    "Lleuque": {
        "Costa / Cordillera de Nahuelbuta": {
            "lat_min": -38.15, "lat_max": -37.55,
            "lon_min": -73.25, "lon_max": -72.75
        },
        "Andes / Maule - Araucanía": {
            "lat_min": -39.30, "lat_max": -35.40,
            "lon_min": -72.00, "lon_max": -70.75
        }
    },
    "Queule": {
        # Plan RECOGE (p. 5): 35°47'S (Cauquenes) hasta 37°41'S (Nahuelbuta/Arauco)
        "Distribución Costera Continua (Maule - Nahuelbuta)": {
            "lat_min": -37.85, "lat_max": -35.70,
            "lon_min": -73.35, "lon_max": -72.60
        }
    },
    "Ruil": {
        # Cordillera de la Costa del Maule (Curepto ~35°06'S hasta Pelluhue/Chanco ~35°50'S)
        "Distribución Costera Continua (Cordillera de la Costa, Maule)": {
            "lat_min": -36.05, "lat_max": -35.00,
            "lon_min": -72.65, "lon_max": -72.00
        }
    }
}

def validar_envolvente_nativa(especie, lat, lon, env):
    '''Valida si una coordenada cae dentro del rango biogeográfico nativo.'''
    if not (env["lat_min"] <= lat <= env["lat_max"]):
        return False
    if not (env["lon_min"] <= lon):
        return False
    # Exclusión fina de registros cultivados en el Valle Central al sur del río Biobío para Queule
    if especie == "Queule":
        lon_limite = -72.95 if lat <= -37.0 else env["lon_max"]
        return lon <= lon_limite
    return lon <= env["lon_max"]

sufijo_escala = "rce_cl" if ESCALA_EVALUACION == "RCE_CHILE" else "uicn_global"
ARCHIVO_SALIDA = f"analisis_completo_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.gpkg"

print(f"Especie seleccionada : {ESPECIE_SELECCIONADA} ({TAXONOMIA_ESPECIES[ESPECIE_SELECCIONADA]})")
print(f"Escala de evaluación : {ESCALA_EVALUACION} | Países GBIF: {PAISES_CONSULTA}")
print(f"AOO oficial UICN/RCE : {GRID_AOO_UICN_M} m ({AREA_CELDA_UICN_KM2:.1f} km²/celda)")
lit_cfg = AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]
if lit_cfg["grid_m"]:
    print(f"AOO fino literatura  : {lit_cfg['grid_m']} m ({(lit_cfg['grid_m']/1000)**2:.2f} km²/celda)")
print(f"Referencia literatura: {lit_cfg['referencia']}")
print(f"Archivo GeoPackage   : {ARCHIVO_SALIDA}")

## 3. Carga de Localidades de Referencia y Sistemas de Referencia Espacial (CRS)

### ¿Por qué usamos dos proyecciones (LAEA *Equal-Area* y UTM Chile)?
1. **Cálculo métrico de áreas y grillas (LAEA *Equal-Area*):**  
   Las proyecciones UTM (`EPSG:32718` / `EPSG:32719`) son **conformes** (preservan ángulos, no áreas exactas) y además Chile continental en el centro-sur cruza el meridiano $-72^\circ$ (límite entre zonas 18S y 19S). Siguiendo el estándar cartográfico de la UICN (*GeoCAT / rCAT*), proyectamos los datos a una **proyección equivalente (*Lambert Azimuthal Equal-Area*, LAEA)** centrada en el centroide $(\phi_0, \lambda_0)$ de la especie:
   $$\text{PROJ: } \texttt{+proj=laea +lat\_0=}\phi_0\;\texttt{+lon\_0=}\lambda_0\;\texttt{+datum=WGS84 +units=m +no\_defs}$$
   Esta proyección garantiza **distorsión de área nula** en el cálculo de EOO, AOO y áreas hexagonales.
2. **Exportación cartográfica para QGIS (UTM WGS84):**  
   Al guardar el GeoPackage (`.gpkg`), las capas se reproyectan al huso UTM chileno correspondiente (`EPSG:32718` si $\bar{\lambda} \le -72^\circ$, o `EPSG:32719` en caso contrario) para compatibilidad inmediata con las capas del curso en QGIS, conservando intactos los atributos de área calculados en LAEA.

In [ ]:
REPO_GITHUB_RAW = "https://raw.githubusercontent.com/magdzraw/byc-2026/main/data"

def resolver_archivo_entrada(ruta_archivo):
    '''Busca el archivo localmente, en data/ o ../data/, y como respaldo lo descarga desde GitHub.'''
    if os.path.exists(ruta_archivo):
        return ruta_archivo
    nombre = os.path.basename(ruta_archivo)
    for cand in (os.path.join("data", nombre), os.path.join("..", "data", nombre)):
        if os.path.exists(cand):
            return cand
    url_raw = f"{REPO_GITHUB_RAW}/{nombre}"
    try:
        r = SESSION_HTTP.get(url_raw, timeout=20)
        if r.status_code == 200:
            with open(nombre, "wb") as f:
                f.write(r.content)
            print(f"Archivo de ejemplo '{nombre}' descargado automáticamente desde GitHub (magdzraw/byc-2026).")
            return nombre
    except Exception:
        pass
    raise FileNotFoundError(
        f"No se encontró '{ruta_archivo}' en el entorno local ni en el repositorio de GitHub. "
        f"En Colab, súbelo al panel de archivos o ajusta la ruta."
    )

def cargar_csv_localidades(ruta_csv):
    '''Carga el CSV de localidades manejando codificación UTF-8/Latin-1 y separador ';' o ','.'''
    ruta_csv = resolver_archivo_entrada(ruta_csv)
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            with open(ruta_csv, "r", encoding=enc) as fh:
                primera_linea = fh.readline()
            sep = ";" if primera_linea.count(";") >= primera_linea.count(",") else ","
            df = pd.read_csv(ruta_csv, sep=sep, encoding=enc)
            df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo decodificar '{ruta_csv}'.")

df_total = cargar_csv_localidades(CSV_UNIFICADO)
df_sitios = df_total[df_total["Especie"].str.strip() == ESPECIE_SELECCIONADA].copy()

if df_sitios.empty:
    raise ValueError(f"No se encontraron localidades para '{ESPECIE_SELECCIONADA}' en {CSV_UNIFICADO}.")

df_sitios["Lat"] = pd.to_numeric(df_sitios["Lat"], errors="coerce")
df_sitios["Lon"] = pd.to_numeric(df_sitios["Lon"], errors="coerce")
df_sitios = df_sitios.dropna(subset=["Lat", "Lon"]).reset_index(drop=True)

# Definición de proyecciones
lat_centro = float(df_sitios["Lat"].mean())
lon_centro = float(df_sitios["Lon"].mean())

CRS_EQUAL_AREA = f"+proj=laea +lat_0={lat_centro:.4f} +lon_0={lon_centro:.4f} +datum=WGS84 +units=m +no_defs"
CRS_EXPORT_UTM = "EPSG:32718" if lon_centro <= -72.0 else "EPSG:32719"

cols_sitios = [c for c in ["ID", "Referencia", "Codigo", "Lat", "Lon"] if c in df_sitios.columns]
gdf_sitios_wgs84 = gpd.GeoDataFrame(
    df_sitios[cols_sitios],
    geometry=gpd.points_from_xy(df_sitios["Lon"], df_sitios["Lat"]),
    crs="EPSG:4326"
)
gdf_sitios_ea = gdf_sitios_wgs84.to_crs(CRS_EQUAL_AREA)

print(f"Localidades de referencia cargadas: {len(gdf_sitios_wgs84)}")
print(f"CRS cálculo métrico (Equal-Area) : LAEA centrado en ({lat_centro:.3f}°, {lon_centro:.3f}°)")
print(f"CRS exportación QGIS             : {CRS_EXPORT_UTM}")
display(df_sitios[cols_sitios])

## 4. Descarga y Curaduría de Ocurrencias de la Especie Focal (GBIF + Terreno)

Para garantizar trazabilidad y calidad de datos conforme a los estándares UICN/RCE:
1. **Resolución taxonómica automática (`taxonKey`):** Consultamos `/v1/species/match` del *GBIF Backbone Taxonomy* para obtener el `usageKey` aceptado. Esto incluye automáticamente todos los sinónimos reconocidos (por ejemplo, *Podocarpus andinus* para *Prumnopitys andina* o *Gomortega nitida* para *Gomortega keule*) sin riesgo de duplicados ni homónimos.
2. **Filtros de calidad espacial y biológica:**
   * `occurrenceStatus = 'PRESENT'` (excluye registros de ausencia explícita).
   * `hasCoordinate = 'true'` y `hasGeospatialIssue = 'false'`.
   * Exclusión de especímenes fósiles (`FOSSIL_SPECIMEN`) y registros declarados como cultivados, introducidos o manejados (`CULTIVATED`, `INTRODUCED`, `MANAGED`).
   * Filtro de precisión espacial: `coordinateUncertaintyInMeters <= INCERTIDUMBRE_MAXIMA_M` (o no reportada, documentándose su proporción).
   * Validación dentro de la envolvente biogeográfica nativa (descarta ejemplares ornamentales en parques urbanos o jardines botánicos).
3. **Integración con localidades de terreno (`Coordenadas_codigo.csv`):** Si `INCLUIR_SITIOS_CSV_EN_UICN = True`, los sitios confirmados de terreno/muestreo molecular se incorporan al conjunto de presencias validadas (identificados en el campo `fuente`).

In [ ]:
URL_GBIF_MATCH = "https://api.gbif.org/v1/species/match"
URL_GBIF_OCC = "https://api.gbif.org/v1/occurrence/search"

nombre_cientifico = TAXONOMIA_ESPECIES[ESPECIE_SELECCIONADA]
resp_match = SESSION_HTTP.get(URL_GBIF_MATCH, params={"name": nombre_cientifico}, timeout=15)
resp_match.raise_for_status()
match_info = resp_match.json()
TAXON_KEY = match_info.get("usageKey")
if not TAXON_KEY:
    raise RuntimeError(f"No se pudo resolver taxonKey en GBIF para '{nombre_cientifico}'.")

print(f"Resolución GBIF Backbone: {match_info.get('scientificName')} | taxonKey = {TAXON_KEY} | estado = {match_info.get('status')}")

envolventes_sp = ENVOLVENTES_SUBPOBLACIONES[ESPECIE_SELECCIONADA]
es_disyunta = len(envolventes_sp) > 1

# Paginación completa de ocurrencias en GBIF
registros_crudos = []
offset = 0
limit = 300
total_gbif = None
TOPE_SEGURIDAD_ESPECIE = 15000

while True:
    params = [
        ("taxonKey", str(TAXON_KEY)),
        ("hasCoordinate", "true"),
        ("hasGeospatialIssue", "false"),
        ("occurrenceStatus", "PRESENT"),
        ("limit", str(limit)),
        ("offset", str(offset)),
    ]
    for pais in PAISES_CONSULTA:
        params.append(("country", pais))

    resp = SESSION_HTTP.get(URL_GBIF_OCC, params=params, timeout=20)
    if resp.status_code != 200:
        print(f"Aviso HTTP {resp.status_code} en offset {offset}; deteniendo paginación.")
        break

    data = resp.json()
    if total_gbif is None:
        total_gbif = data.get("count", 0)
    lote = data.get("results", [])
    if not lote:
        break

    registros_crudos.extend(lote)
    offset += limit
    if data.get("endOfRecords", False) or offset >= total_gbif:
        break
    if offset >= TOPE_SEGURIDAD_ESPECIE:
        print(f"ADVERTENCIA: Se alcanzó el tope de {TOPE_SEGURIDAD_ESPECIE} registros (total GBIF = {total_gbif}).")
        break

# Curaduría de registros con conteo transparente de descartes
descartes = {"fosil": 0, "cultivado": 0, "incertidumbre": 0, "fuera_envolvente": 0}
registros_validos = []
sin_incertidumbre = 0

for r in registros_crudos:
    lat = r.get("decimalLatitude")
    lon = r.get("decimalLongitude")
    if lat is None or lon is None:
        continue

    if r.get("basisOfRecord") == "FOSSIL_SPECIMEN":
        descartes["fosil"] += 1
        continue

    estab = str(r.get("establishmentMeans", "")).upper()
    deg_estab = str(r.get("degreeOfEstablishment", "")).upper()
    if any(k in estab or k in deg_estab for k in ("CULTIVATED", "INTRODUCED", "MANAGED")):
        descartes["cultivado"] += 1
        continue

    incert = r.get("coordinateUncertaintyInMeters")
    if incert is None:
        sin_incertidumbre += 1
    elif incert > INCERTIDUMBRE_MAXIMA_M:
        descartes["incertidumbre"] += 1
        continue

    sub_asignada = None
    for sub_nombre, env in envolventes_sp.items():
        if validar_envolvente_nativa(ESPECIE_SELECCIONADA, lat, lon, env):
            sub_asignada = sub_nombre
            break

    if sub_asignada is None:
        descartes["fuera_envolvente"] += 1
        continue

    registros_validos.append({
        "gbif_id": str(r.get("key", "")),
        "species": r.get("species", nombre_cientifico),
        "subpoblacion": sub_asignada,
        "decimalLatitude": float(lat),
        "decimalLongitude": float(lon),
        "uncertainty_m": -1.0 if incert is None else float(incert),
        "year": r.get("year"),
        "countryCode": r.get("countryCode", "CL"),
        "fuente": "GBIF",
    })

df_sp = pd.DataFrame(registros_validos)

# Integrar localidades validadas del CSV (terreno / muestreo molecular)
if INCLUIR_SITIOS_CSV_EN_UICN:
    filas_csv = []
    for _, fila in df_sitios.iterrows():
        lat, lon = float(fila["Lat"]), float(fila["Lon"])
        sub_asig = None
        for s_nombre, env in envolventes_sp.items():
            if validar_envolvente_nativa(ESPECIE_SELECCIONADA, lat, lon, env):
                sub_asig = s_nombre
                break
        if sub_asig is None:
            sub_asig = list(envolventes_sp.keys())[0]
        filas_csv.append({
            "gbif_id": f"CSV_{fila.get('Codigo', fila.get('ID', ''))}",
            "species": nombre_cientifico,
            "subpoblacion": sub_asig,
            "decimalLatitude": lat,
            "decimalLongitude": lon,
            "uncertainty_m": 0.0,
            "year": 2026,
            "countryCode": "CL",
            "fuente": "Localidad_Referencia",
        })
    df_csv_pts = pd.DataFrame(filas_csv)
    df_sp = pd.concat([df_sp, df_csv_pts], ignore_index=True)

if df_sp.empty:
    raise RuntimeError("No quedaron ocurrencias válidas tras los filtros. Revisa las envolventes o el CSV.")

# Deduplicación espacial exacta (EOO y AOO dependen de sitios únicos de presencia)
n_antes_dedup = len(df_sp)
df_sp["lat_r"] = df_sp["decimalLatitude"].round(5)
df_sp["lon_r"] = df_sp["decimalLongitude"].round(5)
df_sp = df_sp.drop_duplicates(subset=["lat_r", "lon_r"]).drop(columns=["lat_r", "lon_r"]).reset_index(drop=True)

gdf_sp_wgs84 = gpd.GeoDataFrame(
    df_sp,
    geometry=gpd.points_from_xy(df_sp["decimalLongitude"], df_sp["decimalLatitude"]),
    crs="EPSG:4326"
)
gdf_sp_ea = gdf_sp_wgs84.to_crs(CRS_EQUAL_AREA)

print(f"\nResumen de curaduría GBIF ({PAISES_CONSULTA}):")
print(f"  Registros brutos descargados     : {len(registros_crudos)} (reportados por API: {total_gbif})")
print(f"  Descartados por fósil            : {descartes['fosil']}")
print(f"  Descartados por cultivado/intro. : {descartes['cultivado']}")
print(f"  Descartados por incert. >{INCERTIDUMBRE_MAXIMA_M}m : {descartes['incertidumbre']}")
print(f"  Descartados fuera de envolvente  : {descartes['fuera_envolvente']}")
print(f"  Válidos sin incert. reportada    : {sin_incertidumbre}")
print(f"  Sitios únicos finales (con CSV)  : {len(gdf_sp_ea)} (deduplicados desde {n_antes_dedup})")
print("\nDistribución por subpoblación biogeográfica:")
print(gdf_sp_ea["subpoblacion"].value_counts().to_string())

## 5. Máscara Territorial de Alta Resolución (GADM 4.1 / Natural Earth 1:10m)

En Chile centro-sur y austral (Chiloé, fiordos de Palena, Hornopirén, Chaitén y borde costero del Maule–Biobío), las capas mundiales de baja resolución (`1:50m` o `1:110m`) omiten islas y simplifican drásticamente la costa, deformando el área del EOO.

Aquí cargamos automáticamente el límite oficial de **alta resolución de GADM v4.1** (`gadm41_CHL_0.json.zip` para `RCE_CHILE`, sumando `gadm41_ARG_0.json.zip` si `ESCALA_EVALUACION == 'UICN_GLOBAL'`), con respaldo automático a **Natural Earth 1:10m** si el servidor GADM no respondiera.  
*(Si prefieres usar un shapefile local del INE/IDE Chile como `CHL_ADM1.shp`, basta con indicar su ruta en `RUTA_LIMITE_LOCAL`.)*

In [ ]:
RUTA_LIMITE_LOCAL = ""  # Opcional: ruta a CHL_ADM1.shp o similar si se desea usar capa local

def cargar_mascara_alta_resolucion(paises, bbox_wgs84, ruta_local=""):
    '''Descarga y recorta la máscara terrestre de alta resolución (GADM 4.1 o Natural Earth 10m).'''
    gdfs = []
    fuente_usada = ""

    if ruta_local and os.path.exists(ruta_local):
        g = gpd.read_file(ruta_local).to_crs("EPSG:4326")
        gdfs.append(g)
        fuente_usada = f"Archivo local ({ruta_local})"
    else:
        mapa_iso3 = {"CL": "CHL", "AR": "ARG"}
        cache_dir = "cache_limites_geo"
        os.makedirs(cache_dir, exist_ok=True)
        gadm_ok = True

        for p in paises:
            iso3 = mapa_iso3[p]
            zip_local = os.path.join(cache_dir, f"gadm41_{iso3}_0.json.zip")
            url_gadm = f"https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3}_0.json.zip"
            try:
                if not os.path.exists(zip_local):
                    r = SESSION_HTTP.get(url_gadm, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
                    r.raise_for_status()
                    with open(zip_local, "wb") as f:
                        f.write(r.content)
                with zipfile.ZipFile(zip_local, "r") as zf:
                    json_name = [n for n in zf.namelist() if n.endswith(".json")][0]
                    json_local = os.path.join(cache_dir, json_name)
                    if not os.path.exists(json_local):
                        zf.extract(json_name, cache_dir)
                gdfs.append(gpd.read_file(json_local).to_crs("EPSG:4326"))
            except Exception as e:
                print(f"Aviso: No se pudo cargar GADM 4.1 para {iso3} ({e}). Usando respaldo Natural Earth 10m...")
                gadm_ok = False
                gdfs = []
                break

        if gadm_ok and gdfs:
            fuente_usada = f"GADM v4.1 alta resolución ({', '.join(mapa_iso3[p] for p in paises)})"
        else:
            ne_zip = os.path.join(cache_dir, "ne_10m_admin_0_countries.zip")
            url_ne = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
            if not os.path.exists(ne_zip):
                r = SESSION_HTTP.get(url_ne, timeout=45, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                with open(ne_zip, "wb") as f:
                    f.write(r.content)
            ne_gdf = gpd.read_file(f"zip://{ne_zip}").to_crs("EPSG:4326")
            iso_list = [mapa_iso3[p] for p in paises]
            col_iso = "ADM0_A3" if "ADM0_A3" in ne_gdf.columns else "ISO_A3"
            gdfs.append(ne_gdf[ne_gdf[col_iso].isin(iso_list)].copy())
            fuente_usada = f"Natural Earth 1:10m ({', '.join(iso_list)})"

    gdf_paises = pd.concat(gdfs, ignore_index=True)
    gdf_paises["geometry"] = gdf_paises.geometry.make_valid()
    gdf_clip = gpd.clip(gdf_paises, bbox_wgs84)
    gdf_ea = gdf_clip.to_crs(CRS_EQUAL_AREA)
    mascara_geom = shapely.make_valid(gdf_ea.geometry.union_all())
    return mascara_geom, fuente_usada

bbox_estudio_ea = box(*gdf_sp_ea.total_bounds).buffer(30000)
bbox_estudio_wgs84 = gpd.GeoSeries([bbox_estudio_ea], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]

mascara_tierra_ea, FUENTE_MASCARA = cargar_mascara_alta_resolucion(
    PAISES_CONSULTA, bbox_estudio_wgs84, ruta_local=RUTA_LIMITE_LOCAL
)

# Validación espacial estricta contra el polígono terrestre (tolerancia costera de 1 km para deriva GPS)
if ESCALA_EVALUACION == "RCE_CHILE":
    mascara_tol = mascara_tierra_ea.buffer(1000)
    en_territorio = gdf_sp_ea.geometry.within(mascara_tol)
    n_fuera_frontera = int((~en_territorio).sum())
    if n_fuera_frontera > 0:
        print(f"Se depuraron {n_fuera_frontera} registros ubicados fuera del límite nacional de Chile (tolerancia costera 1 km).")
        gdf_sp_ea = gdf_sp_ea[en_territorio].reset_index(drop=True)
        gdf_sp_wgs84 = gdf_sp_ea.to_crs("EPSG:4326")

print(f"Máscara territorial lista: {FUENTE_MASCARA}")
print(f"Ocurrencias definitivas para cálculo EOO/AOO: {len(gdf_sp_ea)}")

## 6. Extensión de Presencia — EOO (Criterio B1 UICN / RCE)

### Definición y Fórmula
La **Extensión de Presencia (EOO)** es el área contenida dentro del límite imaginario continuo más corto que engloba todos los sitios conocidos, inferidos o proyectados de presencia actual de un taxón (UICN v16, Sec. 4.9). Se calcula como el área del **Polígono Convexo Mínimo (MCP, *convex hull*)**:

$$\text{EOO} = \frac{\text{Área}\left(\operatorname{Conv}(P)\right)}{10^6}\quad [\text{km}^2]$$

donde $P = \{p_1, p_2, \dots, p_n\}$ es el conjunto de ocurrencias nativas proyectadas en LAEA *Equal-Area*.

### Reglas Normativas Aplicadas
* **No exclusión de disyunciones en B1 (UICN Sec. 4.9):** Para el Criterio B1, la UICN desaconseja excluir disyunciones (como el Valle Central entre Costa y Andes), ya que el EOO cuantifica la **dispersión espacial del riesgo**. Por ello, el **EOO Oficial B1** corresponde al **MCP global** de todos los registros (recortado a territorio nacional terrestre en modo `RCE_CHILE`; reportando también el MCP bruto sin recorte marino).
* **Análisis biogeográfico complementario:** Para especies con distribución disyunta (*Alerce*, *Araucaria*, *Lleuque*), se calculan además los polígonos separados por núcleo (Costa, Depresión Intermedia, Andes) como capa informativa para conservación regional y cruce con estructura genética.
* **Umbrales geográficos preliminares del Criterio B1:**
  * $\text{EOO} < 100\text{ km}^2 \rightarrow$ umbral de **En Peligro Crítico (CR)**
  * $\text{EOO} < 5.000\text{ km}^2 \rightarrow$ umbral de **En Peligro (EN)**
  * $\text{EOO} < 20.000\text{ km}^2 \rightarrow$ umbral de **Vulnerable (VU)**
  * **¡Importante!** Cumplir el umbral geográfico B1 **no asigna por sí solo una categoría de amenaza**: la UICN/RCE exige cumplir además **al menos 2 de 3 subcriterios**: *(a)* fragmentación severa o número de localidades por amenaza ($\le 1, 5, 10$), *(b)* disminución continua, o *(c)* fluctuaciones extremas. Tampoco superar $20.000\text{ km}^2$ implica automáticamente *Preocupación Menor (LC)*, pues la especie puede calificar bajo otros criterios (p. ej., **Criterio A** por reducción poblacional, como ocurre en Alerce y Araucaria).

In [ ]:
def evaluar_umbral_b1(km2):
    '''Clasifica únicamente el umbral geográfico preliminar de EOO (Criterio B1).'''
    if km2 < 100:
        return "Umbral B1 de CR (<100 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 5000:
        return "Umbral B1 de EN (<5.000 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 20000:
        return "Umbral B1 de VU (<20.000 km²; requiere ≥2 subcriterios a/b/c)"
    else:
        return "Supera umbral B1 (≥20.000 km²; no activa B1, evaluar criterios A/B2/C/D)"

def construir_mcp(geometrias_gdf, mascara=None):
    '''Construye el Polígono Convexo Mínimo (MCP) y opcionalmente recorta por máscara terrestre.'''
    n = len(geometrias_gdf)
    if n >= 3:
        geom = geometrias_gdf.geometry.union_all().convex_hull
    elif n in (1, 2):
        geom = geometrias_gdf.geometry.buffer(2000).union_all()
    else:
        return Polygon()

    if mascara is not None:
        geom = shapely.make_valid(geom.intersection(mascara))
        if geom.geom_type == "GeometryCollection":
            polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
            geom = unary_union(polys) if polys else Polygon()
    return geom

# 1. EOO MCP Bruto (sin recorte marino — referencia estricta UICN v16 Sec. 4.9)
eoo_mcp_bruto_geom = construir_mcp(gdf_sp_ea, mascara=None)
eoo_mcp_bruto_km2 = round(eoo_mcp_bruto_geom.area / 1e6, 2)

# 2. EOO MCP Recortado a máscara terrestre nacional / regional (estándar práctico fichas RCE Chile)
eoo_mcp_tierra_geom = construir_mcp(gdf_sp_ea, mascara=mascara_tierra_ea)
eoo_mcp_tierra_km2 = round(eoo_mcp_tierra_geom.area / 1e6, 2)

# EOO oficial según la escala elegida
if ESCALA_EVALUACION == "RCE_CHILE":
    eoo_oficial_geom = eoo_mcp_tierra_geom
    eoo_oficial_km2 = eoo_mcp_tierra_km2
    metodo_eoo = "MCP Nacional Chile recortado a tierra firme (RCE-MMA)"
else:
    eoo_oficial_geom = eoo_mcp_bruto_geom
    eoo_oficial_km2 = eoo_mcp_bruto_km2
    metodo_eoo = "MCP Global sin exclusión de disyunciones (UICN v16 Sec. 4.9)"

gdf_eoo_oficial_ea = gpd.GeoDataFrame([{
    "especie": ESPECIE_SELECCIONADA,
    "escala": ESCALA_EVALUACION,
    "metodo_eoo": metodo_eoo,
    "n_registros": len(gdf_sp_ea),
    "eoo_km2": eoo_oficial_km2,
    "eoo_mcp_bruto_km2": eoo_mcp_bruto_km2,
    "eoo_mcp_tierra_km2": eoo_mcp_tierra_km2,
    "umbral_preliminar_b1": evaluar_umbral_b1(eoo_oficial_km2),
    "geometry": eoo_oficial_geom,
}], crs=CRS_EQUAL_AREA)

# 3. Análisis biogeográfico complementario por núcleos/subpoblaciones (solo si la especie es disyunta)
gdf_eoo_subpob_ea = None
if es_disyunta:
    filas_sub = []
    for sub_nombre in envolventes_sp.keys():
        pts_sub = gdf_sp_ea[gdf_sp_ea["subpoblacion"] == sub_nombre]
        if len(pts_sub) > 0:
            geom_sub = construir_mcp(pts_sub, mascara=mascara_tierra_ea)
            if not geom_sub.is_empty:
                area_sub = round(geom_sub.area / 1e6, 2)
                filas_sub.append({
                    "especie": ESPECIE_SELECCIONADA,
                    "subpoblacion": sub_nombre,
                    "tipo": "Núcleo biogeográfico (complementario, no reemplaza B1)",
                    "n_registros": len(pts_sub),
                    "area_nucleo_km2": area_sub,
                    "geometry": geom_sub,
                })
    if filas_sub:
        gdf_eoo_subpob_ea = gpd.GeoDataFrame(filas_sub, crs=CRS_EQUAL_AREA)

print(f"EOO Oficial ({ESCALA_EVALUACION}) : {eoo_oficial_km2:,.2f} km²")
print(f"  ├─ MCP bruto (sin recorte marino) : {eoo_mcp_bruto_km2:,.2f} km²")
print(f"  ├─ MCP terrestre ({FUENTE_MASCARA}) : {eoo_mcp_tierra_km2:,.2f} km²")
print(f"  └─ Evaluación preliminar B1       : {evaluar_umbral_b1(eoo_oficial_km2)}")

if gdf_eoo_subpob_ea is not None:
    suma_nucleos = gdf_eoo_subpob_ea["area_nucleo_km2"].sum()
    matriz_intermedia = max(0.0, eoo_mcp_tierra_km2 - suma_nucleos)
    print("\nDesglose biogeográfico complementario por núcleos (no sustituye B1):")
    for _, r in gdf_eoo_subpob_ea.iterrows():
        print(f"  ├─ {r['subpoblacion']}: {r['area_nucleo_km2']:,.2f} km² ({r['n_registros']} registros)")
    print(f"  └─ Suma núcleos: {suma_nucleos:,.2f} km² | Matriz intermedia excluida: {matriz_intermedia:,.2f} km²")

## 7. Área de Ocupación — AOO (Criterio B2 UICN / RCE + Escala Fina de Literatura)

### Definición, Escala Estándar y Optimización de Grilla
El **Área de Ocupación (AOO)** es el área dentro del EOO efectivamente ocupada por el taxón (UICN v16, Sec. 4.10). Se estima superponiendo una cuadrícula regular de lado $g$ y contando las celdas ocupadas $N_{\text{celdas}}$:

$$\text{AOO}(g) = \min_{(\delta_x, \delta_y) \in [0, g)^2} \left|\left\{\left(\left\lfloor \frac{x_i - \delta_x}{g} \right\rfloor,\; \left\lfloor \frac{y_i - \delta_y}{g} \right\rfloor\right) : (x_i, y_i) \in P\right\}\right| \times \left(\frac{g}{1000}\right)^2\quad [\text{km}^2]$$

* **Grilla obligatoria para Criterio B2 ($g = 2000\text{ m}$, celda $= 4\text{ km}^2$):** La UICN (Sec. 4.10) y el RCE-MMA calibran los umbrales de B2 exclusivamente a esta escala:
  * $\text{AOO} < 10\text{ km}^2 \rightarrow$ umbral de **En Peligro Crítico (CR)**
  * $\text{AOO} < 500\text{ km}^2 \rightarrow$ umbral de **En Peligro (EN)**
  * $\text{AOO} < 2.000\text{ km}^2 \rightarrow$ umbral de **Vulnerable (VU)**
* **Minimización por desplazamiento de origen $(\delta_x, \delta_y)$:** Como el conteo de celdas depende de dónde se ubique el origen de la malla, evaluamos $10 \times 10 = 100$ desplazamientos sistemáticos y seleccionamos el **número mínimo de celdas ocupadas**, según indica explícitamente la UICN (Sec. 4.10).
* **Grilla fina complementaria de literatura / Plan RECOGE ($g = 1000\text{ m}$ para Queule y Ruil):** Permite comparar directamente contra estudios de micro-fragmentación (p. ej., **Plan RECOGE Queule 2022, p. 6**, que usa celdas de $1\text{ km}^2$), manteniéndola separada del Criterio B2 oficial.

In [ ]:
def evaluar_umbral_b2(km2):
    '''Clasifica únicamente el umbral geográfico preliminar de AOO en 2x2 km (Criterio B2).'''
    if km2 < 10:
        return "Umbral B2 de CR (<10 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 500:
        return "Umbral B2 de EN (<500 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 2000:
        return "Umbral B2 de VU (<2.000 km²; requiere ≥2 subcriterios a/b/c)"
    else:
        return "Supera umbral B2 (≥2.000 km²; no activa B2, evaluar criterios A/B1/C/D)"

def calcular_aoo_optimizado(gdf_pts_ea, grid_m, pasos=10):
    '''
    Calcula el AOO buscando el origen de grilla (dx, dy) que minimiza el número de celdas
    ocupadas (UICN v16 Sec. 4.10). Devuelve GeoDataFrame de celdas y estadísticas.
    '''
    xs = gdf_pts_ea.geometry.x.to_numpy()
    ys = gdf_pts_ea.geometry.y.to_numpy()
    area_celda_km2 = (grid_m / 1000.0) ** 2

    # Conteo base sin optimizar (origen 0, 0) para comparación didáctica
    celdas_base = len(set(zip(np.floor(xs / grid_m).astype(int), np.floor(ys / grid_m).astype(int))))

    mejor_n = None
    mejor_offset = (0.0, 0.0)
    mejores_celdas = None

    desplazamientos = np.linspace(0.0, grid_m, pasos, endpoint=False)
    for dx in desplazamientos:
        c_idx = np.floor((xs - dx) / grid_m).astype(int)
        for dy in desplazamientos:
            r_idx = np.floor((ys - dy) / grid_m).astype(int)
            unicas = set(zip(c_idx, r_idx))
            n_u = len(unicas)
            if mejor_n is None or n_u < mejor_n:
                mejor_n = n_u
                mejor_offset = (float(dx), float(dy))
                mejores_celdas = unicas

    dx_opt, dy_opt = mejor_offset
    poligonos = [
        box(
            c * grid_m + dx_opt,
            r * grid_m + dy_opt,
            (c + 1) * grid_m + dx_opt,
            (r + 1) * grid_m + dy_opt,
        )
        for (c, r) in sorted(mejores_celdas)
    ]

    aoo_km2 = round(mejor_n * area_celda_km2, 2)
    gdf_celdas = gpd.GeoDataFrame({
        "especie": ESPECIE_SELECCIONADA,
        "taxon": nombre_cientifico,
        "escala": ESCALA_EVALUACION,
        "celda_id": range(1, len(poligonos) + 1),
        "resolucion_m": grid_m,
        "area_celda_km2": area_celda_km2,
        "n_celdas_min": mejor_n,
        "n_celdas_origen0": celdas_base,
        "aoo_total_km2": aoo_km2,
        "geometry": poligonos,
    }, crs=CRS_EQUAL_AREA)

    return gdf_celdas, {
        "n_celdas_min": mejor_n,
        "n_celdas_origen0": celdas_base,
        "aoo_km2": aoo_km2,
        "area_celda_km2": area_celda_km2,
        "offset_m": mejor_offset,
    }

# 1. AOO Oficial UICN / RCE (2x2 km obligatorio)
gdf_aoo_uicn_ea, stats_aoo_uicn = calcular_aoo_optimizado(
    gdf_sp_ea, grid_m=GRID_AOO_UICN_M, pasos=PASOS_OPTIMIZACION_AOO
)
gdf_aoo_uicn_ea["umbral_preliminar_b2"] = evaluar_umbral_b2(stats_aoo_uicn["aoo_km2"])

print(f"AOO Oficial UICN/RCE (2x2 km): {stats_aoo_uicn['aoo_km2']:,.2f} km² "
      f"({stats_aoo_uicn['n_celdas_min']} celdas mínimas; vs. {stats_aoo_uicn['n_celdas_origen0']} en origen fijo 0,0)")
print(f"  └─ Evaluación preliminar B2 : {evaluar_umbral_b2(stats_aoo_uicn['aoo_km2'])}")

# 2. AOO Complementario de Escala Fina según Literatura / RECOGE (si aplica)
gdf_aoo_lit_ea = None
stats_aoo_lit = None
grid_lit_m = AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]["grid_m"]

if grid_lit_m is not None and grid_lit_m != GRID_AOO_UICN_M:
    gdf_aoo_lit_ea, stats_aoo_lit = calcular_aoo_optimizado(
        gdf_sp_ea, grid_m=grid_lit_m, pasos=PASOS_OPTIMIZACION_AOO
    )
    gdf_aoo_lit_ea["nota_escala"] = "Escala fina de literatura/RECOGE (no aplicar umbrales B2)"
    print(f"\nAOO Complementario Literatura ({grid_lit_m}x{grid_lit_m} m): {stats_aoo_lit['aoo_km2']:,.2f} km² "
          f"({stats_aoo_lit['n_celdas_min']} celdas mínimas; vs. {stats_aoo_lit['n_celdas_origen0']} en origen 0,0)")
    print(f"  └─ Referencia: {AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]['referencia']}")

## 8. Biodiversidad Local de Flora Vascular Acompañante (`Tracheophyta`) en Malla Hexagonal

### 8.1. ¿Cómo y con qué datos se construyen los hexágonos?

Para evaluar la diversidad de flora vascular en el entorno inmediato de las poblaciones estudiadas, construimos una **teselación hexagonal regular** en la proyección equivalente (**LAEA *Equal-Area***), garantizando que todas las unidades de muestreo tengan exactamente la misma superficie.

> **¿Por qué hexágonos y no cuadrados?** En ecología del paisaje, los hexágonos ofrecen **6 vecinos equidistantes** desde su centro (evitando la ambigüedad entre vecinos ortogonales y diagonales propia de las cuadrículas cuadradas) y una menor relación perímetro/área, reduciendo efectos de borde al agrupar ocurrencias puntuales.

#### Paso a paso de la construcción geométrica
1. **Datos base (centros espaciales):** Se utilizan las coordenadas de las **localidades de referencia** cargadas desde `Coordenadas_codigo.csv` (`gdf_sitios_ea`, proyectadas en metros en LAEA *Equal-Area*).
2. **Área de influencia local (`BUFFER_LOCAL_M = 7500 m`):** Alrededor de cada localidad se genera un círculo (*buffer*) de radio $7{,}5\text{ km}$ y todos los círculos se fusionan en una sola geometría (`area_estudio_local`) que delimita el paisaje circundante a los sitios de estudio.
3. **Geometría de cada hexágono regular (`RADIO_HEXAGONO_M = 2500 m`):**
   * Cada celda es un hexágono regular en orientación ***pointy-top*** (vértices superior e inferior en punta) con circunradio $r = \texttt{RADIO\_HEXAGONO\_M} = 2500\text{ m}$ (distancia del centro a cada vértice, igual a la longitud de cada lado).
   * **Área exacta de cada hexágono (`area_hex_km2`):**
     $$A_{\text{hex}} = \frac{3\sqrt{3}}{2}\, r^2$$
     Con $r = 2{,}5\text{ km}$, cada hexágono cubre exactamente $A_{\text{hex}} \approx 16{,}24\text{ km}^2$.
   * **Coordenadas de los 6 vértices** respecto al centro $(x_c, y_c)$ de cada celda (`offsets_x`, `offsets_y`):
     $$x_k = x_c + r \cos(\theta_k), \qquad y_k = y_c + r \sin(\theta_k), \qquad \theta_k = \frac{\pi}{6} + k\frac{\pi}{3} \quad (k = 0, 1, \dots, 5)$$
     El ángulo inicial $\pi/6 = 30^\circ$ rota la figura a orientación *pointy-top* (se usan 7 puntos en `np.linspace` para cerrar el polígono volviendo al vértice inicial).
   * **Espaciado entre centros para encaje tipo panal (sin huecos ni solapamientos):**
     * **Paso horizontal entre columnas (`dx`):** $dx = \sqrt{3}\, r \approx 4330{,}1\text{ m}$ (distancia entre lados opuestos del hexágono).
     * **Paso vertical entre filas (`dy`):** $dy = 1{,}5\, r = 3750\text{ m}$.
     * **Desfase alternado (`offset_x`):** en las filas impares (`fila % 2 == 1`), todos los centros se desplazan horizontalmente $dx / 2 = \frac{\sqrt{3}}{2} r$ para que las filas encajen exactamente entre sí.
4. **Selección de hexágonos del área de estudio:** Se recorre el rectángulo envolvente (`xmin, ymin, xmax, ymax`) de `area_estudio_local` y se conservan únicamente los hexágonos que **intersectan** dicho buffer (`poly.intersects(area_estudio_local)`).

---

### 8.2. Datos biológicos incorporados a cada hexágono (GBIF `Tracheophyta`)
* **Filtro taxonómico (`phylumKey = 7707728`):** Se consulta la API v1 de GBIF filtrando exclusivamente por el filo **Tracheophyta** (plantas vasculares: árboles, arbustos, hierbas y helechos). *(Nota técnica: GBIF exige la clave numérica `phylumKey`; el parámetro textual `phylum` es ignorado por la API y mezclaría fauna y hongos).*
* **Cobertura exacta y paginación:** Los hexágonos adyacentes se fusionan en parches conectados (con una tolerancia de $1\text{ m}$ para cerrar micro-brechas de punto flotante) y se consulta GBIF usando el *bounding box* real en WGS84 de cada parche con paginación completa (`offset += 300`), aplicando `occurrenceStatus = 'PRESENT'`, excluyendo fósiles, cultivados/introducidos e incertidumbre $> \texttt{INCERTIDUMBRE\_MAXIMA\_M}$, y deduplicando por `gbif_id`.
* **Asignación espacial:** Mediante una intersección punto-en-polígono (`gpd.sjoin(..., predicate="within")`), cada registro de flora vascular se vincula al `hex_id` que lo contiene.

---

### 8.3. Variables e Índices de Diversidad y Completitud Calculados por Hexágono

#### Variables base dentro de cada hexágono
| Símbolo | Columna / Código | Significado |
| :---: | :--- | :--- |
| $N$ | `n_registros` | Número total de registros de flora vascular dentro del hexágono ($N = \sum_{i=1}^{S_{\text{obs}}} n_i$). |
| $n_i$ | `conteos` | Número de registros correspondientes a la $i$-ésima especie dentro del hexágono. |
| $p_i$ | `p_i = conteos / N` | Proporción (frecuencia relativa) de registros de la especie $i$ respecto al total $N$ ($\sum_{i=1}^{S_{\text{obs}}} p_i = 1$). |
| $S_{\text{obs}}$ | `riqueza_s` | **Riqueza observada:** número de especies vasculares distintas registradas en el hexágono. |
| $f_1$ | `(conteos == 1).sum()` | Número de especies ***singletons*** (registradas exactamente $1$ vez en el hexágono, $n_i = 1$). |
| $f_2$ | `(conteos == 2).sum()` | Número de especies ***doubletons*** (registradas exactamente $2$ veces en el hexágono, $n_i = 2$). |
| `confiable` | `1` si $N \ge \texttt{N\_MIN\_ESFUERZO}$ | Bandera de esfuerzo mínimo (por defecto $N \ge 15$) para evitar interpretar índices calculados sobre muy pocos registros. |

#### Fórmulas e interpretación ecológica
1. **Índice de diversidad de Shannon-Wiener ($H'$, `shannon_h`):**
   $$H' = -\sum_{i=1}^{S_{\text{obs}}} p_i \ln(p_i)$$
   Combina riqueza y uniformidad (expresado en *nats* al usar logaritmo natural $\ln$). Vale $0$ cuando solo hay una especie y aumenta al crecer el número de especies y la equitatividad entre ellas (en comunidades vegetales suele oscilar entre $1{,}5$ y $3{,}5$).

2. **Índice de equitatividad de Pielou ($J'$, `pielou_j`):**
   $$J' = \begin{cases} \dfrac{H'}{\ln(S_{\text{obs}})} & \text{si } S_{\text{obs}} > 1 \\ \text{NaN (indeterminado)} & \text{si } S_{\text{obs}} \le 1 \end{cases}$$
   Normaliza $H'$ respecto a la diversidad máxima posible $\ln(S_{\text{obs}})$. Varía entre $0$ (dominancia extrema de una especie) y $1$ (todas las especies igual de representadas). Cuando $S_{\text{obs}} \le 1$, el cociente $0/0$ es indeterminado y se reporta como `NaN`.

3. **Índice de Gini-Simpson ($1 - D$, `simpson_1_d`):**
   $$1 - D = 1 - \sum_{i=1}^{S_{\text{obs}}} p_i^2$$
   Representa la probabilidad de que dos registros seleccionados al azar dentro del hexágono pertenezcan a especies diferentes. Varía entre $0$ y $1 - 1/S_{\text{obs}}$ y otorga mayor peso a las especies más abundantes/frecuentes que Shannon.

4. **Estimador no paramétrico de riqueza Chao1 ($\hat{S}_{\text{Chao1}}$, `chao1_s`):**
   $$\hat{S}_{\text{Chao1}} = S_{\text{obs}} + \frac{f_1 (f_1 - 1)}{2 (f_2 + 1)}$$
   Estima la riqueza total esperada (especies detectadas más una cota inferior de especies no detectadas) utilizando la proporción de especies raras ($f_1$ y $f_2$). Se emplea la forma corregida por sesgo ($f_2 + 1$ en el denominador), que permanece definida incluso si $f_2 = 0$.

5. **Cobertura de muestra de Chao & Jost (2012) ($\hat{C}$, `cobertura_c`):**
   $$\hat{C} = 1 - \frac{f_1}{N}\left[\frac{(N - 1)\, f_1}{(N - 1)\, f_1 + 2 f_2}\right]$$
   Estima la proporción ($0$ a $1$) de la comunidad representada por las especies ya observadas en el hexágono. Valores cercanos a $1$ indican un inventario más completo; valores bajos ($< 0{,}5$) advierten submuestreo.

> **Nota metodológica para la discusión:** Los datos de GBIF provienen de fuentes heterogéneas (herbarios y ciencia ciudadana) y **no** de parcelas estandarizadas de abundancia. Por ello, $H'$, $J'$ y $1-D$ deben interpretarse como **diversidad aparente basada en registros**, siempre analizada en conjunto con el esfuerzo muestral ($N$) y la cobertura de muestra ($\hat{C}$).

In [ ]:
# 1. Generación de malla hexagonal regular (orientación pointy-top) en LAEA Equal-Area
# Buffer circular de radio BUFFER_LOCAL_M (7500 m) alrededor de cada localidad de referencia
area_estudio_local = gdf_sitios_ea.geometry.buffer(BUFFER_LOCAL_M).union_all()

# Espaciado entre centros para teselación continua sin huecos ni solapamientos:
dx = np.sqrt(3) * RADIO_HEXAGONO_M  # Paso horizontal entre centros de una misma fila (~4330 m)
dy = 1.5 * RADIO_HEXAGONO_M         # Paso vertical entre filas consecutivas (3750 m)

# Límites del rectángulo envolvente alineados a la grilla
xmin, ymin, xmax, ymax = area_estudio_local.bounds
x_start = np.floor(xmin / dx) * dx
y_start = np.floor(ymin / dy) * dy

# Desplazamientos relativos de los 6 vértices (pi/6 = 30° deja vértices arriba/abajo: pointy-top)
angulos = np.linspace(np.pi / 6, 2 * np.pi + np.pi / 6, 7)
offsets_x = RADIO_HEXAGONO_M * np.cos(angulos)
offsets_y = RADIO_HEXAGONO_M * np.sin(angulos)

hex_geoms = []
y_act = y_start - dy
fila = 0
while y_act <= ymax + dy:
    # Desfase alternado de dx/2 en filas impares para encaje tipo panal
    offset_x = (dx / 2) if (fila % 2 == 1) else 0.0
    x_act = x_start - dx + offset_x
    while x_act <= xmax + dx:
        poly = Polygon([(x_act + ox, y_act + oy) for ox, oy in zip(offsets_x, offsets_y)])
        # Se conservan solo los hexágonos que intersectan el área de influencia de las localidades
        if poly.intersects(area_estudio_local):
            hex_geoms.append(poly)
        x_act += dx
    y_act += dy
    fila += 1

# Área exacta de cada hexágono regular: A_hex = (3*sqrt(3)/2) * r^2
area_hex_km2 = round((3 * np.sqrt(3) / 2) * ((RADIO_HEXAGONO_M / 1000.0) ** 2), 3)
gdf_hex_ea = gpd.GeoDataFrame(
    {"hex_id": range(1, len(hex_geoms) + 1), "area_hex_km2": area_hex_km2, "geometry": hex_geoms},
    crs=CRS_EQUAL_AREA
)

# 2. Parches conectados de hexágonos -> bounding boxes exactos en WGS84 para GBIF
# (buffer de 1 m fusiona bordes adyacentes evitando micro-brechas de punto flotante)
union_hex = gdf_hex_ea.geometry.buffer(1.0).union_all()
parches_ea = list(union_hex.geoms) if union_hex.geom_type == "MultiPolygon" else [union_hex]
gdf_parches_wgs84 = gpd.GeoDataFrame(geometry=parches_ea, crs=CRS_EQUAL_AREA).to_crs("EPSG:4326")

PHYLUM_KEY_TRACHEOPHYTA = 7707728
registros_flora = []
parches_truncados = 0

print(f"Malla generada: {len(gdf_hex_ea)} hexágonos ({area_hex_km2:.2f} km² c/u) agrupados en {len(gdf_parches_wgs84)} parches.")
print(f"Descargando flora vascular (Tracheophyta, phylumKey={PHYLUM_KEY_TRACHEOPHYTA}) con paginación completa...")

for p_idx, geom_p in enumerate(gdf_parches_wgs84.geometry, start=1):
    minx, miny, maxx, maxy = geom_p.bounds
    margen = 0.005
    lat_rng = f"{miny - margen:.5f},{maxy + margen:.5f}"
    lon_rng = f"{minx - margen:.5f},{maxx + margen:.5f}"

    offset = 0
    limit = 300
    total_parche = None

    while True:
        params = [
            ("decimalLatitude", lat_rng),
            ("decimalLongitude", lon_rng),
            ("phylumKey", str(PHYLUM_KEY_TRACHEOPHYTA)),
            ("hasCoordinate", "true"),
            ("hasGeospatialIssue", "false"),
            ("occurrenceStatus", "PRESENT"),
            ("limit", str(limit)),
            ("offset", str(offset)),
        ]
        if ESCALA_EVALUACION == "RCE_CHILE":
            params.append(("country", "CL"))

        try:
            resp = SESSION_HTTP.get(URL_GBIF_OCC, params=params, timeout=15)
            if resp.status_code != 200:
                break
            data = resp.json()
        except Exception:
            break

        if total_parche is None:
            total_parche = data.get("count", 0)
        lote = data.get("results", [])
        if not lote:
            break

        for r in lote:
            if r.get("basisOfRecord") == "FOSSIL_SPECIMEN":
                continue
            estab = str(r.get("establishmentMeans", "")).upper()
            deg_estab = str(r.get("degreeOfEstablishment", "")).upper()
            if any(k in estab or k in deg_estab for k in ("CULTIVATED", "INTRODUCED", "MANAGED")):
                continue
            incert = r.get("coordinateUncertaintyInMeters")
            if incert is not None and incert > INCERTIDUMBRE_MAXIMA_M:
                continue
            sp_name = r.get("species")
            if sp_name and r.get("decimalLatitude") is not None and r.get("decimalLongitude") is not None:
                registros_flora.append({
                    "gbif_id": str(r.get("key", "")),
                    "phylum": r.get("phylum", "Tracheophyta"),
                    "family": r.get("family", ""),
                    "species": sp_name,
                    "scientificName": r.get("scientificName", sp_name),
                    "lat": float(r["decimalLatitude"]),
                    "lon": float(r["decimalLongitude"]),
                })

        offset += limit
        if data.get("endOfRecords", False) or offset >= total_parche:
            break
        if offset >= MAX_REGISTROS_FLORA_PARCHE:
            parches_truncados += 1
            print(f"  Aviso: Parche {p_idx} alcanzó el tope de {MAX_REGISTROS_FLORA_PARCHE} registros (total disponible en GBIF: {total_parche}).")
            break

# 3. Deduplicación por gbif_id y cálculo de índices ecológicos
df_flora = pd.DataFrame(registros_flora)
if not df_flora.empty:
    df_flora = df_flora.drop_duplicates(subset=["gbif_id"]).reset_index(drop=True)
    gdf_flora_ea = gpd.GeoDataFrame(
        df_flora[["gbif_id", "phylum", "family", "species", "scientificName"]],
        geometry=gpd.points_from_xy(df_flora["lon"], df_flora["lat"]),
        crs="EPSG:4326"
    ).to_crs(CRS_EQUAL_AREA)

    # Restringir a puntos que efectivamente caen dentro de los hexágonos
    join_pts = gpd.sjoin(gdf_flora_ea, gdf_hex_ea[["hex_id", "geometry"]], how="inner", predicate="within")
    gdf_flora_dentro_ea = gdf_flora_ea.loc[join_pts.index.unique()].copy()

    def calcular_diversidad_hex(group):
        conteos = group["species"].value_counts()
        N = int(conteos.sum())
        S = int(len(conteos))
        p_i = conteos / N

        H = float(-np.sum(p_i * np.log(p_i))) if S > 1 else 0.0
        J = float(H / np.log(S)) if S > 1 else np.nan
        simpson = float(1.0 - np.sum(p_i ** 2))

        f1 = int((conteos == 1).sum())
        f2 = int((conteos == 2).sum())
        chao1 = float(S + (f1 * (f1 - 1)) / (2.0 * (f2 + 1))) if S > 0 else 0.0

        if N > 1 and f1 > 0:
            denom = (N - 1) * f1 + 2 * f2
            cobertura = float(1.0 - (f1 / N) * (((N - 1) * f1) / denom)) if denom > 0 else 0.0
        elif N > 0 and f1 == 0:
            cobertura = 1.0
        else:
            cobertura = np.nan

        return pd.Series({
            "n_registros": N,
            "riqueza_s": S,
            "shannon_h": round(H, 3),
            "pielou_j": round(J, 3) if pd.notna(J) else np.nan,
            "simpson_1_d": round(simpson, 3),
            "chao1_s": round(chao1, 2),
            "cobertura_c": round(cobertura, 3) if pd.notna(cobertura) else np.nan,
            "confiable": 1 if N >= N_MIN_ESFUERZO else 0,
            "sp_dominante": conteos.index[0],
        })

    if not join_pts.empty:
        metricas_hex = join_pts.groupby("hex_id").apply(calcular_diversidad_hex, include_groups=False).reset_index()
        gdf_hex_final_ea = gdf_hex_ea.merge(metricas_hex, on="hex_id", how="left")
        gdf_hex_final_ea = gdf_hex_final_ea.fillna({
            "n_registros": 0,
            "riqueza_s": 0,
            "shannon_h": 0.0,
            "simpson_1_d": 0.0,
            "chao1_s": 0.0,
            "confiable": 0,
            "sp_dominante": "Sin registros",
        })
        gdf_hex_final_ea["n_registros"] = gdf_hex_final_ea["n_registros"].astype(int)
        gdf_hex_final_ea["riqueza_s"] = gdf_hex_final_ea["riqueza_s"].astype(int)
        gdf_hex_final_ea["confiable"] = gdf_hex_final_ea["confiable"].astype(int)
    else:
        gdf_flora_dentro_ea = gpd.GeoDataFrame(columns=["gbif_id", "phylum", "family", "species", "scientificName", "geometry"], crs=CRS_EQUAL_AREA)
        gdf_hex_final_ea = gdf_hex_ea.assign(
            n_registros=0, riqueza_s=0, shannon_h=0.0, pielou_j=np.nan,
            simpson_1_d=0.0, chao1_s=0.0, cobertura_c=np.nan, confiable=0, sp_dominante="Sin registros"
        )
else:
    gdf_flora_dentro_ea = gpd.GeoDataFrame(columns=["gbif_id", "phylum", "family", "species", "scientificName", "geometry"], crs=CRS_EQUAL_AREA)
    gdf_hex_final_ea = gdf_hex_ea.assign(
        n_registros=0, riqueza_s=0, shannon_h=0.0, pielou_j=np.nan,
        simpson_1_d=0.0, chao1_s=0.0, cobertura_c=np.nan, confiable=0, sp_dominante="Sin registros"
    )

hex_con_datos = gdf_hex_final_ea[gdf_hex_final_ea["n_registros"] > 0]
hex_confiables = gdf_hex_final_ea[gdf_hex_final_ea["confiable"] == 1]

print(f"\nResumen de flora vascular acompañante (Tracheophyta):")
print(f"  Registros únicos dentro de hexágonos : {len(gdf_flora_dentro_ea)}")
print(f"  Especies vasculares registradas      : {gdf_flora_dentro_ea['species'].nunique() if not gdf_flora_dentro_ea.empty else 0}")
print(f"  Hexágonos con datos / confiables     : {len(hex_con_datos)} / {len(hex_confiables)} (umbral N ≥ {N_MIN_ESFUERZO})")
if not hex_confiables.empty:
    print(f"  Shannon (H') promedio (confiables)   : {hex_confiables['shannon_h'].mean():.3f}")
    print(f"  Pielou (J') promedio (confiables)    : {hex_confiables['pielou_j'].mean():.3f}")
    print(f"  Simpson (1-D) promedio (confiables)  : {hex_confiables['simpson_1_d'].mean():.3f}")
    print(f"  Cobertura (C) promedio (confiables)  : {hex_confiables['cobertura_c'].mean():.3f}")

## 9. Exportación a GeoPackage (`.gpkg` para QGIS) y Reporte Consolidado

Se eliminan versiones previas del archivo `.gpkg` para evitar capas huérfanas y se exportan todas las capas reproyectadas a **UTM Chile (`EPSG:32718` o `EPSG:32719`)**, conservando intactas las áreas calculadas en LAEA *Equal-Area*:

| Capa en el GeoPackage | Contenido |
| :--- | :--- |
| `iucn_eoo_oficial` | Polígono EOO oficial B1 (MCP nacional Chile en modo `RCE_CHILE`, o MCP global en `UICN_GLOBAL`). |
| `eoo_subpoblaciones_biogeograficas` | Polígonos separados por núcleo (Costa / Depresión Intermedia / Andes) para especies disyuntas. |
| `iucn_aoo_2x2km_oficial` | Grilla estándar UICN/RCE de $2 \times 2\text{ km}$ optimizada (mínimo de celdas) para Criterio B2. |
| `aoo_escala_fina_literatura` | Grilla complementaria ($1 \times 1\text{ km}$) para especies con referencia RECOGE (*Queule*, *Ruil*). |
| `ocurrencias_especie_validadas` | Registros curados de GBIF + localidades de terreno utilizadas en EOO y AOO. |
| `localidades_referencia` | Sitios del estudio (`ID`, `Referencia`, `Codigo`, `Lat`, `Lon`). |
| `hex_biodiversidad_local` | Malla hexagonal con $N$, $S_{\text{obs}}$, $H'$, $J'$, $1-D$, $S_{\text{Chao1}}$, $\hat{C}$ y especie vascular dominante. |
| `ocurrencias_flora_tracheophyta` | Registros de flora vascular acompañante (`phylumKey=7707728`) dentro de los hexágonos. |

In [ ]:
if os.path.exists(ARCHIVO_SALIDA):
    os.remove(ARCHIVO_SALIDA)

# Reproyección a UTM Chile para interoperabilidad con capas de QGIS
gdf_hex_final_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="hex_biodiversidad_local", driver="GPKG")
gdf_sitios_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="localidades_referencia", driver="GPKG")
gdf_eoo_oficial_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="iucn_eoo_oficial", driver="GPKG")
if gdf_eoo_subpob_ea is not None:
    gdf_eoo_subpob_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="eoo_subpoblaciones_biogeograficas", driver="GPKG")
gdf_aoo_uicn_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="iucn_aoo_2x2km_oficial", driver="GPKG")
if gdf_aoo_lit_ea is not None:
    gdf_aoo_lit_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="aoo_escala_fina_literatura", driver="GPKG")
gdf_sp_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="ocurrencias_especie_validadas", driver="GPKG")
if not gdf_flora_dentro_ea.empty:
    gdf_flora_dentro_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="ocurrencias_flora_tracheophyta", driver="GPKG")

# Tabla resumen lista para informes y presentaciones
resumen_rows = [
    ("Especie / Taxón", f"{ESPECIE_SELECCIONADA} ({nombre_cientifico})"),
    ("Escala de evaluación", f"{ESCALA_EVALUACION} ({', '.join(PAISES_CONSULTA)})"),
    ("Ocurrencias nativas únicas", f"{len(gdf_sp_ea)} (GBIF + CSV)" if INCLUIR_SITIOS_CSV_EN_UICN else f"{len(gdf_sp_ea)} (GBIF)"),
    ("EOO Oficial (Criterio B1)", f"{eoo_oficial_km2:,.2f} km² — {evaluar_umbral_b1(eoo_oficial_km2)}"),
    ("EOO MCP bruto (sin recorte marino)", f"{eoo_mcp_bruto_km2:,.2f} km²"),
    ("AOO Oficial 2×2 km (Criterio B2)", f"{stats_aoo_uicn['aoo_km2']:,.2f} km² ({stats_aoo_uicn['n_celdas_min']} celdas mín.) — {evaluar_umbral_b2(stats_aoo_uicn['aoo_km2'])}"),
]
if stats_aoo_lit is not None:
    resumen_rows.append((
        f"AOO Literatura ({grid_lit_m}×{grid_lit_m} m)",
        f"{stats_aoo_lit['aoo_km2']:,.2f} km² ({stats_aoo_lit['n_celdas_min']} celdas mín.; complementario, no B2)"
    ))
resumen_rows.extend([
    ("Referencia oficial / literatura", AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]["referencia"]),
    ("Flora vascular (Tracheophyta)", f"{len(gdf_flora_dentro_ea)} registros | {gdf_flora_dentro_ea['species'].nunique() if not gdf_flora_dentro_ea.empty else 0} especies en {len(hex_con_datos)} hexágonos"),
    ("GeoPackage exportado (CRS)", f"{ARCHIVO_SALIDA} ({CRS_EXPORT_UTM}; áreas calculadas en LAEA Equal-Area)"),
])

df_resumen = pd.DataFrame(resumen_rows, columns=["Parámetro", "Resultado"])
print(f"\n================ REPORTE CONSOLIDADO ({ESPECIE_SELECCIONADA} — {ESCALA_EVALUACION}) ================")
for k, v in resumen_rows:
    print(f"  {k:<36}: {v}")
if ESCALA_EVALUACION == "RCE_CHILE" and ESPECIE_SELECCIONADA in ("Alerce", "Araucaria", "Lleuque"):
    print("\n  [Nota RCE Paso 2]: Especie compartida con Argentina. Para clasificación regional definitiva,")
    print("  evaluar conectividad y eventual efecto de rescate desde poblaciones argentinas (UICN Regional v4.0).")

display(df_resumen)

## 10. Visualización Interactiva con Selector de Capas (`Folium`)

Utiliza el panel de capas (esquina superior derecha del mapa) para alternar los **mapas base** (compatibles con Google Colab sin bloqueo de tiles ni API key: **Mapa Claro CartoDB**, **Topográfico Esri** y **Satélite Esri**) y activar o desactivar independientemente:
* Malla hexagonal de diversidad de flora vascular ($H'$ de Shannon)
* EOO Oficial (B1) y núcleos biogeográficos complementarios
* AOO Oficial $2 \times 2\text{ km}$ (B2) y AOO fino de literatura ($1 \times 1\text{ km}$, cuando aplica)
* Ocurrencias validadas y localidades de referencia (`Codigo` y `Referencia`)

In [ ]:
centro_mapa = [lat_centro, lon_centro]
# tiles=None evita el tile por defecto de OpenStreetMap (bloqueado en iframes de Google Colab)
m = folium.Map(location=centro_mapa, zoom_start=8, tiles=None)

# Mapas base compatibles con Google Colab (sin API key ni bloqueo por Referer)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    name="Mapa Claro (CartoDB Positron)",
    show=True,
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ, USGS, NASA",
    name="Topográfico (Esri World Topo)",
    show=False,
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Source: Esri, Maxar, Earthstar Geographics",
    name="Satélite (Esri World Imagery)",
    show=False,
).add_to(m)

# 1. Malla hexagonal de biodiversidad (Shannon H')
gdf_hex_final_ea.to_crs("EPSG:4326").explore(
    m=m,
    name="Biodiversidad Flora Vascular (Shannon H')",
    column="shannon_h",
    cmap="viridis",
    tooltip=["hex_id", "n_registros", "riqueza_s", "shannon_h", "pielou_j", "simpson_1_d", "chao1_s", "cobertura_c", "confiable", "sp_dominante"],
    popup=True,
    legend=True,
    style_kwds={"fillOpacity": 0.55, "weight": 0.8},
)

# 2. EOO Oficial (B1)
gdf_eoo_oficial_ea.to_crs("EPSG:4326").explore(
    m=m,
    name=f"EOO Oficial B1 ({eoo_oficial_km2:,.0f} km²)",
    color="darkorange",
    style_kwds={"fill": True, "fillOpacity": 0.12, "weight": 2.5},
    tooltip=["especie", "escala", "eoo_km2", "umbral_preliminar_b1"],
)

# 3. Núcleos biogeográficos (si es disyunta)
if gdf_eoo_subpob_ea is not None:
    gdf_eoo_subpob_ea.to_crs("EPSG:4326").explore(
        m=m,
        name="Núcleos Biogeográficos (Complementario)",
        color="royalblue",
        style_kwds={"fill": True, "fillOpacity": 0.12, "weight": 2.0, "dashArray": "5, 5"},
        tooltip=["subpoblacion", "n_registros", "area_nucleo_km2"],
        show=False,
    )

# 4. AOO Oficial 2x2 km (B2)
gdf_aoo_uicn_ea.to_crs("EPSG:4326").explore(
    m=m,
    name=f"AOO Oficial 2×2 km ({stats_aoo_uicn['aoo_km2']:,.0f} km²)",
    color="crimson",
    style_kwds={"fillOpacity": 0.45, "weight": 1.0},
    tooltip=["celda_id", "resolucion_m", "aoo_total_km2", "umbral_preliminar_b2"],
)

# 5. AOO Escala Fina de Literatura (cuando aplica, ej. Queule / Ruil)
if gdf_aoo_lit_ea is not None:
    gdf_aoo_lit_ea.to_crs("EPSG:4326").explore(
        m=m,
        name=f"AOO Literatura {grid_lit_m}×{grid_lit_m} m ({stats_aoo_lit['aoo_km2']:,.0f} km²)",
        color="purple",
        style_kwds={"fillOpacity": 0.55, "weight": 1.0},
        tooltip=["celda_id", "resolucion_m", "aoo_total_km2", "nota_escala"],
        show=False,
    )

# 6. Ocurrencias validadas de la especie focal
gdf_sp_ea.to_crs("EPSG:4326").explore(
    m=m,
    name=f"Ocurrencias {ESPECIE_SELECCIONADA} ({len(gdf_sp_ea)})",
    color="forestgreen",
    marker_type="circle_marker",
    marker_kwds={"radius": 3.5, "fill_opacity": 0.8},
    tooltip=["species", "subpoblacion", "fuente", "year", "uncertainty_m"],
    show=False,
)

# 7. Localidades de referencia
tooltip_sitios = [c for c in ["Codigo", "Referencia", "ID"] if c in gdf_sitios_ea.columns]
gdf_sitios_ea.to_crs("EPSG:4326").explore(
    m=m,
    name="Localidades de Referencia",
    color="black",
    marker_type="circle_marker",
    marker_kwds={"radius": 6, "fill_opacity": 1.0},
    tooltip=tooltip_sitios,
)

folium.LayerControl(collapsed=False).add_to(m)
mapa_base_html = f"mapa_interactivo_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.html"
m.save(mapa_base_html)
print(f"Mapa interactivo guardado como archivo independiente: '{mapa_base_html}'")
display(m)

---

## 11. Módulo Opcional: Espacialización de Estructura Genética Poblacional (`STRUCTURE`)

Este módulo integra los resultados de agrupamiento bayesiano generados en **STRUCTURE** (Pritchard et al., 2000; Falush et al., 2003) directamente con las localidades de referencia (`Coordenadas_codigo.csv`), el EOO/AOO y los mapas interactivos y de publicación.

### 11.1. Marco Metodológico y Fórmulas

1. **Matriz de ancestría ($Q$):** Para un número asumido de grupos genéticos $K$, STRUCTURE estima la proporción de ancestría $q_{ik} \in [0, 1]$ de cada individuo $i$ en cada *cluster* $k$ ($\sum_{k=1}^{K} q_{ik} = 1$), y el promedio poblacional $\bar{Q}_{pk}$ para cada localidad $p \in \{1, \dots, P\}$.
2. **Selección del $K$ óptimo — Método de Evanno et al. (2005):**  
   A partir del archivo `.zip` con todas las corridas (`*_f`) de la carpeta `Results/` de STRUCTURE, sea $L_r(K) = \ln P(D)$ la verosimilitud estimada en la réplica $r$ para un valor de $K$, con media $\overline{L}(K)$ y desviación estándar muestral $s(L(K))$:
   $$L'(K) = \overline{L}(K) - \overline{L}(K - 1), \qquad |L''(K)| = \left|L'(K + 1) - L'(K)\right| = \left|\overline{L}(K + 1) - 2\overline{L}(K) + \overline{L}(K - 1)\right|$$
   $$\Delta K = \frac{|L''(K)|}{s\!\left(L(K)\right)}$$
   > **Nota:** $\Delta K$ no está definido para $K = 1$ ni para $K_{\max}$, y tiende a capturar el nivel jerárquico más alto de estructuración (frecuentemente $K = 2$). Por ello, este módulo permite **inspeccionar tanto $\Delta K$ como $\overline{L}(K) \pm s$** y elegir **manualmente** cualquier `K_SELECCIONADO` con sentido biológico/biogeográfico.
3. **Consenso entre réplicas (alineación tipo CLUMPP):** Para un mismo $K$, distintas iteraciones pueden permutar el número asignado a cada cluster (*label switching*). El modo `CONSENSO_CLUMPP` alinea automáticamente todas las réplicas contra la corrida de mayor $\ln P(D)$ mediante el algoritmo de asignación óptima húngara (`scipy.optimize.linear_sum_assignment`), promedia las matrices $Q$ alineadas y ordena los clusters geográficamente de **Norte a Sur** para que los colores fluyan de manera coherente en el mapa y el barplot.
4. **Métricas poblacionales derivadas:**
   * **Ancestría dominante:** $Q_{\max, p} = \max_{k} \bar{Q}_{pk}$. Se clasifica operacionalmente cada localidad como **Pura** ($Q_{\max} \ge 0{,}80$) o **Zona de mezcla / transición** ($Q_{\max} < 0{,}80$).
   * **Entropía de mezcla genética:** $H_{Q, p} = -\sum_{k=1}^{K} \bar{Q}_{pk} \ln(\bar{Q}_{pk})$ (vale $0$ si la población pertenece $100\%$ a un único cluster y aumenta con la mezcla).
5. **Discusión crítica sobre interpolación espacial IDW (*tipo `tess3r`*):**  
   Dado que los individuos comparten la coordenada de su **localidad de referencia** ($P = 8\text{–}12$ sitios), una interpolación continua por distancia inversa ponderada ($w_j = d_j^{-p}$) asume continuidad isotrópica y puede suavizar artificialmente barreras reales (como el Valle Central en especies disyuntas) o extrapolar fuera del muestreo. Por ello, **los diagramas de torta discretos por localidad son la representación principal recomendada**, mientras que el IDW se ofrece **apagado por defecto (`MOSTRAR_IDW_ANCESTRIA = False`)** con doble enmascaramiento (recortado a los núcleos/EOO terrestre y a un radio máximo alrededor de las localidades muestreadas) y opacidad modulada por la certeza $\frac{Q_{\max} - 1/K}{1 - 1/K}$.

In [ ]:
#@title Parámetros del Módulo 11: Estructura Genética (STRUCTURE) { display-mode: "form" }
ACTIVAR_MODULO_STRUCTURE = True  #@param {type:"boolean"}
ARCHIVO_STRUCTURE = "Results_lleuque.zip"  #@param {type:"string"}
K_SELECCIONADO = 4  #@param {type:"integer"}
MODO_REPLICAS = "CONSENSO_CLUMPP"  #@param ["CONSENSO_CLUMPP", "MEJOR_LNPD"]
ORDEN_BARPLOT = "ID_ORIGINAL"  #@param ["NORTE_A_SUR", "ID_ORIGINAL"]
MOSTRAR_IDW_ANCESTRIA = True  #@param {type:"boolean"}
RADIO_MASCARA_IDW_KM = 60  #@param {type:"integer"}
POTENCIA_IDW = 2.0  #@param {type:"number"}

# Paleta de alto contraste y accesible para daltonismo (los dos primeros tonos evocan púrpura/verde clásico)
PALETA_CLUSTERS = [
    "#4B0055",  # K1: Púrpura oscuro
    "#1B7837",  # K2: Verde bosque
    "#D95F02",  # K3: Naranjo quemado
    "#2166AC",  # K4: Azul profundo
    "#E6AB02",  # K5: Mostaza / dorado
    "#B2182B",  # K6: Carmesí
    "#01665E",  # K7: Verde petróleo
    "#7570B3",  # K8: Lavanda oscuro
    "#A65628",  # K9: Terracota
    "#666666",  # K10: Gris grafito
]

def parsear_archivo_structure_f(texto, nombre_archivo="run_f"):
    '''Parsea un archivo crudo *_f de STRUCTURE 2.3.x devolviendo metadatos, Q poblacional y Q individual.'''
    m_k = re.search(r"^\s*(\d+)\s+populations assumed", texto, re.MULTILINE)
    m_lnpd = re.search(r"Estimated Ln Prob of Data\s*=\s*([-\d\.]+)", texto)
    if not m_k or not m_lnpd:
        return None

    K = int(m_k.group(1))
    lnpd = float(m_lnpd.group(1))
    m_alpha = re.search(r"Mean value of alpha\s*=\s*([-\d\.]+)", texto)
    alpha = float(m_alpha.group(1)) if m_alpha else np.nan
    m_burn = re.search(r"^\s*(\d+)\s+Burn-in period", texto, re.MULTILINE)
    m_reps = re.search(r"^\s*(\d+)\s+Reps", texto, re.MULTILINE)
    fsts = {int(k): float(v) for k, v in re.findall(r"Mean value of Fst_(\d+)\s*=\s*([-\d\.]+)", texto)}

    # 1. Tabla poblacional (Proportion of membership of each pre-defined population)
    pop_rows = []
    in_pop = False
    for line in texto.splitlines():
        if "Given" in line and "Inferred Clusters" in line:
            in_pop = True
            continue
        if in_pop:
            if line.strip().startswith("---"):
                if pop_rows:
                    break
                continue
            m_pop = re.match(r"^\s*(\d+)\s*:\s*(.+)$", line)
            if m_pop:
                pop_id = int(m_pop.group(1))
                nums = re.findall(r"[-+]?\d*\.\d+|\d+", m_pop.group(2))
                if len(nums) >= K + 1:
                    q_vals = [float(x) for x in nums[:K]]
                    n_ind = int(float(nums[K]))
                    pop_rows.append({"Pop": pop_id, "n_ind": n_ind, **{f"K{i+1}": q_vals[i] for i in range(K)}})

    # 2. Tabla individual (Inferred ancestry of individuals)
    ind_rows = []
    in_ind = False
    for line in texto.splitlines():
        if "Inferred ancestry of individuals:" in line:
            in_ind = True
            continue
        if in_ind:
            s = line.strip()
            if not s:
                if ind_rows:
                    break
                continue
            if "Label" in s and "Pop:" in s:
                continue
            m_ind = re.match(r"^\s*(\d+)\s+(\S+)\s+\((\d+)\)\s+(\d+)\s*:\s*(.+)$", line)
            if m_ind:
                ind_id = int(m_ind.group(1))
                label = m_ind.group(2)
                pct_miss = int(m_ind.group(3))
                pop_id = int(m_ind.group(4))
                # Extrae únicamente los K valores principales (ignorando intervalos entre paréntesis si existieran)
                sin_paren = re.sub(r"\([^)]*\)", "", m_ind.group(5))
                nums = re.findall(r"[-+]?\d*\.\d+|\d+", sin_paren)
                if len(nums) >= K:
                    q_vals = [float(x) for x in nums[:K]]
                    ind_rows.append({
                        "ind_id": ind_id, "label": label, "pct_miss": pct_miss, "Pop": pop_id,
                        **{f"K{i+1}": q_vals[i] for i in range(K)}
                    })

    if not ind_rows and not pop_rows:
        return None

    df_ind = pd.DataFrame(ind_rows)
    df_pop = pd.DataFrame(pop_rows)
    if df_pop.empty and not df_ind.empty:
        k_cols = [f"K{i+1}" for i in range(K)]
        df_pop = df_ind.groupby("Pop", as_index=False)[k_cols].mean()
        df_pop["n_ind"] = df_ind.groupby("Pop").size().values

    return {
        "archivo": nombre_archivo,
        "K": K,
        "lnpd": lnpd,
        "alpha": alpha,
        "burnin": int(m_burn.group(1)) if m_burn else None,
        "reps": int(m_reps.group(1)) if m_reps else None,
        "fsts": fsts,
        "df_pop": df_pop,
        "df_ind": df_ind,
    }

def calcular_evanno(runs):
    '''Calcula la tabla de Evanno et al. (2005): Mean LnP(D), SD, L'(K), |L''(K)| y Delta K.'''
    df_runs = pd.DataFrame([{"K": r["K"], "lnpd": r["lnpd"], "archivo": r["archivo"]} for r in runs])
    res = df_runs.groupby("K", as_index=False).agg(
        n_reps=("lnpd", "count"),
        mean_lnpd=("lnpd", "mean"),
        sd_lnpd=("lnpd", lambda x: float(np.std(x, ddof=1)) if len(x) > 1 else np.nan),
        max_lnpd=("lnpd", "max"),
    ).sort_values("K").reset_index(drop=True)

    res["L_prime"] = np.nan
    res["abs_L_double_prime"] = np.nan
    res["delta_K"] = np.nan

    ks = res["K"].tolist()
    means = res["mean_lnpd"].tolist()
    sds = res["sd_lnpd"].tolist()

    for i in range(len(ks)):
        if i >= 1 and ks[i] - ks[i - 1] == 1:
            res.loc[i, "L_prime"] = means[i] - means[i - 1]
    for i in range(1, len(ks) - 1):
        if ks[i] - ks[i - 1] == 1 and ks[i + 1] - ks[i] == 1:
            l_dd = abs(means[i + 1] - 2.0 * means[i] + means[i - 1])
            res.loc[i, "abs_L_double_prime"] = l_dd
            if pd.notna(sds[i]) and sds[i] > 0:
                res.loc[i, "delta_K"] = l_dd / sds[i]

    return res

def alinear_consenso_clumpp(runs_k, K, modo="CONSENSO_CLUMPP"):
    '''Alinea réplicas de un mismo K mediante asignación húngara óptima (tipo CLUMPP) y calcula consenso.'''
    k_cols = [f"K{i+1}" for i in range(K)]
    runs_ord = sorted(runs_k, key=lambda r: r["lnpd"], reverse=True)
    ref = runs_ord[0]

    if modo == "MEJOR_LNPD" or len(runs_ord) == 1 or K == 1:
        return ref["df_pop"].copy(), ref["df_ind"].copy(), 1.0, ref

    Q_ref_ind = ref["df_ind"][k_cols].to_numpy(dtype=float)
    Q_ref_pop = ref["df_pop"][k_cols].to_numpy(dtype=float)

    ind_acum = Q_ref_ind.copy()
    pop_acum = Q_ref_pop.copy()
    correlaciones = []

    for r in runs_ord[1:]:
        Q_ind = r["df_ind"][k_cols].to_numpy(dtype=float)
        Q_pop = r["df_pop"][k_cols].to_numpy(dtype=float)
        if Q_ind.shape != Q_ref_ind.shape:
            continue
        # Matriz de costo L1 entre clusters de referencia y de la réplica
        cost = np.zeros((K, K), dtype=float)
        for a in range(K):
            for b in range(K):
                cost[a, b] = np.mean(np.abs(Q_ref_ind[:, a] - Q_ind[:, b]))
        _, col_perm = linear_sum_assignment(cost)
        Q_ind_al = Q_ind[:, col_perm]
        Q_pop_al = Q_pop[:, col_perm]
        ind_acum += Q_ind_al
        pop_acum += Q_pop_al
        sim = 1.0 - float(np.mean(np.abs(Q_ref_ind - Q_ind_al)))
        correlaciones.append(sim)

    n_usadas = 1 + len(correlaciones)
    Q_ind_mean = ind_acum / n_usadas
    Q_ind_mean = Q_ind_mean / np.clip(Q_ind_mean.sum(axis=1, keepdims=True), 1e-12, None)
    Q_pop_mean = pop_acum / n_usadas
    Q_pop_mean = Q_pop_mean / np.clip(Q_pop_mean.sum(axis=1, keepdims=True), 1e-12, None)

    df_ind_out = ref["df_ind"].copy()
    df_pop_out = ref["df_pop"].copy()
    for i, col in enumerate(k_cols):
        df_ind_out[col] = np.round(Q_ind_mean[:, i], 4)
        df_pop_out[col] = np.round(Q_pop_mean[:, i], 4)

    sim_media = float(np.mean(correlaciones)) if correlaciones else 1.0
    return df_pop_out, df_ind_out, sim_media, ref

# Estado global del módulo
STRUCTURE_LISTO = False
df_evanno = None
gdf_struct_pop_ea = None
gdf_struct_pop_wgs84 = None
df_struct_ind = None
K_ACTIVO = None
K_COLS_ACTIVOS = []
COLORES_ACTIVOS = []

if not ACTIVAR_MODULO_STRUCTURE:
    print("Módulo 11 desactivado (ACTIVAR_MODULO_STRUCTURE = False).")
    print("Para activarlo: marca ACTIVAR_MODULO_STRUCTURE, sube tu .zip de Results/ de STRUCTURE (o archivo *_f) e indica su nombre en ARCHIVO_STRUCTURE.")
else:
    ruta_str = ARCHIVO_STRUCTURE.strip()
    if not ruta_str:
        zips_cand = [f for f in os.listdir(".") if f.lower().endswith(".zip") and "gadm" not in f.lower() and "ne_10m" not in f.lower()]
        if zips_cand:
            ruta_str = zips_cand[0]
            print(f"Auto-detectado archivo de resultados: {ruta_str}")
        else:
            raise FileNotFoundError("Indica el nombre del archivo .zip (o *_f / .csv) en ARCHIVO_STRUCTURE.")
    ruta_str = resolver_archivo_entrada(ruta_str)

    runs_parseados = []
    es_csv_manual = ruta_str.lower().endswith(".csv")

    if es_csv_manual:
        df_csv_q = cargar_csv_localidades(ruta_str)
        k_cols_csv = [c for c in df_csv_q.columns if re.match(r"^[Kk]\d+$", str(c).strip())]
        if not k_cols_csv or "Pop" not in df_csv_q.columns:
            raise ValueError("El CSV debe contener la columna 'Pop' y columnas de ancestría K1, K2, ..., Kk.")
        k_cols_csv = sorted(k_cols_csv, key=lambda c: int(c[1:]))
        K_ACTIVO = len(k_cols_csv)
        K_COLS_ACTIVOS = [f"K{i+1}" for i in range(K_ACTIVO)]
        renombre = {old: new for old, new in zip(k_cols_csv, K_COLS_ACTIVOS)}
        df_struct_ind = df_csv_q.rename(columns=renombre).copy()
        for c in K_COLS_ACTIVOS:
            df_struct_ind[c] = pd.to_numeric(df_struct_ind[c], errors="coerce")
        df_struct_ind["Pop"] = pd.to_numeric(df_struct_ind["Pop"], errors="coerce").astype(int)
        if "ind_id" not in df_struct_ind.columns:
            df_struct_ind["ind_id"] = range(1, len(df_struct_ind) + 1)
        df_pop_cons = df_struct_ind.groupby("Pop", as_index=False)[K_COLS_ACTIVOS].mean()
        df_pop_cons["n_ind"] = df_struct_ind.groupby("Pop").size().values
        sim_consenso = 1.0
        print(f"CSV de STRUCTURE cargado: {len(df_struct_ind)} individuos, {len(df_pop_cons)} poblaciones, K = {K_ACTIVO}.")
    else:
        if zipfile.is_zipfile(ruta_str):
            with zipfile.ZipFile(ruta_str, "r") as zf:
                for member in zf.namelist():
                    base = os.path.basename(member)
                    if not base or "__MACOSX" in member or member.endswith("/"):
                        continue
                    if base.endswith("_f") or "_run_" in base:
                        try:
                            raw_txt = zf.read(member).decode("utf-8", errors="ignore")
                            parsed = parsear_archivo_structure_f(raw_txt, nombre_archivo=base)
                            if parsed is not None:
                                runs_parseados.append(parsed)
                        except Exception:
                            continue
        else:
            raw_txt = open(ruta_str, "r", encoding="utf-8", errors="ignore").read()
            parsed = parsear_archivo_structure_f(raw_txt, nombre_archivo=os.path.basename(ruta_str))
            if parsed is not None:
                runs_parseados.append(parsed)

        if not runs_parseados:
            raise ValueError(f"No se encontraron archivos válidos '*_f' de STRUCTURE dentro de '{ruta_str}'.")

        ks_disponibles = sorted(set(r["K"] for r in runs_parseados))
        print(f"Archivos STRUCTURE parseados: {len(runs_parseados)} corridas | Valores de K disponibles: {ks_disponibles}")

        df_evanno = calcular_evanno(runs_parseados)
        k_evanno_opt = None
        if df_evanno["delta_K"].notna().any():
            idx_max_dk = df_evanno["delta_K"].idxmax()
            k_evanno_opt = int(df_evanno.loc[idx_max_dk, "K"])
            print(f"K óptimo sugerido por Evanno (Delta K máximo): K = {k_evanno_opt} (Delta K = {df_evanno.loc[idx_max_dk, 'delta_K']:.2f})")
            display(df_evanno.round(3))

        if K_SELECCIONADO > 0:
            if K_SELECCIONADO not in ks_disponibles:
                raise ValueError(f"K_SELECCIONADO={K_SELECCIONADO} no existe en el archivo. K disponibles: {ks_disponibles}")
            K_ACTIVO = K_SELECCIONADO
            print(f"Usando K elegido manualmente: K = {K_ACTIVO}")
        else:
            K_ACTIVO = k_evanno_opt if k_evanno_opt is not None else max(ks_disponibles)
            print(f"Usando K seleccionado automáticamente: K = {K_ACTIVO}")

        runs_k = [r for r in runs_parseados if r["K"] == K_ACTIVO]
        K_COLS_ACTIVOS = [f"K{i+1}" for i in range(K_ACTIVO)]
        df_pop_cons, df_struct_ind, sim_consenso, ref_run = alinear_consenso_clumpp(
            runs_k, K_ACTIVO, modo=MODO_REPLICAS
        )
        print(f"Modo réplicas: {MODO_REPLICAS} ({len(runs_k)} réplicas para K={K_ACTIVO} | similitud media alineada = {sim_consenso:.4f})")

    # Cruce secuencial 1..P con localidades de Coordenadas_codigo.csv (inmune a IDs duplicados como en Ruil)
    df_sit_map = df_sitios.reset_index(drop=True).copy()
    df_sit_map["Pop"] = range(1, len(df_sit_map) + 1)
    if "Codigo" not in df_sit_map.columns:
        df_sit_map["Codigo"] = df_sit_map["Referencia"].astype(str).str.slice(0, 3).str.upper()

    df_pop_join = df_sit_map[["Pop", "ID", "Codigo", "Referencia", "Lat", "Lon"]].merge(
        df_pop_cons, on="Pop", how="inner"
    )
    if df_pop_join.empty:
        raise ValueError("No coincidieron los números de población de STRUCTURE con las localidades del CSV.")

    # Reordenar clusters K1..Kk de Norte a Sur según latitud promedio ponderada por ancestría
    if K_ACTIVO > 1:
        lat_ponderada = []
        for col in K_COLS_ACTIVOS:
            pesos = df_pop_join[col].to_numpy(dtype=float)
            lats = df_pop_join["Lat"].to_numpy(dtype=float)
            lp = float(np.sum(pesos * lats) / np.sum(pesos)) if np.sum(pesos) > 0 else -90.0
            lat_ponderada.append((col, lp))
        # Mayor latitud (menos negativa en hemisferio sur) = más al Norte
        orden_cols = [c for c, _ in sorted(lat_ponderada, key=lambda x: x[1], reverse=True)]
        Q_pop_reord = df_pop_join[orden_cols].to_numpy(dtype=float)
        Q_ind_reord = df_struct_ind[orden_cols].to_numpy(dtype=float)
        for i, col in enumerate(K_COLS_ACTIVOS):
            df_pop_join[col] = np.round(Q_pop_reord[:, i], 4)
            df_struct_ind[col] = np.round(Q_ind_reord[:, i], 4)

    # Métricas poblacionales de ancestría y mezcla
    Q_mat = df_pop_join[K_COLS_ACTIVOS].to_numpy(dtype=float)
    dom_idx = np.argmax(Q_mat, axis=1)
    q_max = np.max(Q_mat, axis=1)
    q_safe = np.clip(Q_mat, 1e-12, 1.0)
    entropia_q = -np.sum(Q_mat * np.log(q_safe), axis=1)

    df_pop_join["cluster_dom"] = [f"K{i+1}" for i in dom_idx]
    df_pop_join["q_max"] = np.round(q_max, 3)
    df_pop_join["entropia_q"] = np.round(entropia_q, 3)
    df_pop_join["estado_mezcla"] = np.where(q_max >= 0.80, "Pura (Qmax ≥ 0.80)", "Mezcla (Qmax < 0.80)")

    # Vincular metadatos de localidad a la tabla individual
    df_struct_ind = df_struct_ind.merge(
        df_pop_join[["Pop", "Codigo", "Referencia", "Lat", "Lon"]], on="Pop", how="left"
    )
    Q_ind_mat = df_struct_ind[K_COLS_ACTIVOS].to_numpy(dtype=float)
    df_struct_ind["cluster_dom"] = [f"K{i+1}" for i in np.argmax(Q_ind_mat, axis=1)]
    df_struct_ind["q_max"] = np.round(np.max(Q_ind_mat, axis=1), 3)

    gdf_struct_pop_wgs84 = gpd.GeoDataFrame(
        df_pop_join,
        geometry=gpd.points_from_xy(df_pop_join["Lon"], df_pop_join["Lat"]),
        crs="EPSG:4326"
    )
    gdf_struct_pop_ea = gdf_struct_pop_wgs84.to_crs(CRS_EQUAL_AREA)

    gdf_struct_ind_ea = gpd.GeoDataFrame(
        df_struct_ind,
        geometry=gpd.points_from_xy(df_struct_ind["Lon"], df_struct_ind["Lat"]),
        crs="EPSG:4326"
    ).to_crs(CRS_EQUAL_AREA)

    # Exportar capas genéticas al mismo GeoPackage (UTM Chile) y tabla Evanno
    gdf_struct_pop_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="structure_poblaciones_q", driver="GPKG")
    gdf_struct_ind_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="structure_individuos_q", driver="GPKG")
    if df_evanno is not None:
        archivo_evanno = f"evanno_resumen_{ESPECIE_SELECCIONADA.lower()}.csv"
        df_evanno.to_csv(archivo_evanno, index=False, sep=";")
        print(f"Tabla Evanno guardada en: {archivo_evanno}")

    COLORES_ACTIVOS = [PALETA_CLUSTERS[i % len(PALETA_CLUSTERS)] for i in range(K_ACTIVO)]

    # Precalcular superficie IDW exploratoria si está activada (así queda lista tanto para 11.2 como para 11.3)
    IDW_RGBA_WGS84 = None
    IDW_BOUNDS_WGS84 = None
    if MOSTRAR_IDW_ANCESTRIA:
        if es_disyunta and gdf_eoo_subpob_ea is not None:
            _dom_idw = gdf_eoo_subpob_ea.geometry.union_all().union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        else:
            _dom_idw = eoo_mcp_tierra_geom.union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        _m_rad = gdf_struct_pop_ea.geometry.buffer(RADIO_MASCARA_IDW_KM * 1000.0).union_all()
        _m_idw = shapely.make_valid(_dom_idw.intersection(_m_rad))
        if not _m_idw.is_empty:
            _m_wgs = gpd.GeoSeries([_m_idw], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]
            _lon0, _lat0, _lon1, _lat1 = _m_wgs.bounds
            _ng = 240
            _GLON, _GLAT = np.meshgrid(np.linspace(_lon0, _lon1, _ng), np.linspace(_lat0, _lat1, _ng))
            _pts_g = gpd.GeoSeries(gpd.points_from_xy(_GLON.ravel(), _GLAT.ravel()), crs="EPSG:4326").to_crs(CRS_EQUAL_AREA)
            _gx = _pts_g.x.to_numpy().reshape(_GLON.shape)
            _gy = _pts_g.y.to_numpy().reshape(_GLAT.shape)
            _in_m = shapely.contains_xy(_m_idw, _gx, _gy)
            if np.any(_in_m):
                _px = gdf_struct_pop_ea.geometry.x.to_numpy()
                _py = gdf_struct_pop_ea.geometry.y.to_numpy()
                _Qm = gdf_struct_pop_ea[K_COLS_ACTIVOS].to_numpy(dtype=float)
                _rgb_p = np.array([mcolors.to_rgb(c) for c in COLORES_ACTIVOS], dtype=np.float32)
                _dist = np.hypot(_gx[_in_m][:, None] - _px[None, :], _gy[_in_m][:, None] - _py[None, :])
                _w = 1.0 / np.power(_dist + 250.0, POTENCIA_IDW)
                _w /= np.sum(_w, axis=1, keepdims=True)
                _Qi = _w @ _Qm
                _dom = np.argmax(_Qi, axis=1)
                _qmax = np.max(_Qi, axis=1)
                _cert = np.clip((_qmax - 1.0 / K_ACTIVO) / max(1.0 - 1.0 / K_ACTIVO, 1e-6), 0.0, 1.0) if K_ACTIVO > 1 else np.ones_like(_qmax)
                _rgba = np.zeros((_ng, _ng, 4), dtype=np.float32)
                _rgba[_in_m, :3] = _rgb_p[_dom]
                _rgba[_in_m, 3] = 0.15 + 0.47 * _cert
                IDW_RGBA_WGS84 = _rgba
                IDW_BOUNDS_WGS84 = [[_lat0, _lon0], [_lat1, _lon1]]
                print(f"Superficie IDW exploratoria lista (radio máximo {RADIO_MASCARA_IDW_KM} km).")

    STRUCTURE_LISTO = True

    print(f"Capas 'structure_poblaciones_q' y 'structure_individuos_q' añadidas a '{ARCHIVO_SALIDA}'.")
    display(df_pop_join[["Pop", "Codigo", "Referencia", "n_ind"] + K_COLS_ACTIVOS + ["cluster_dom", "q_max", "estado_mezcla"]])

### 11.2. Figura Compuesta de Publicación (Panel A: Mapa con Tortas y Líneas Guía | Panel B: Evanno $\Delta K$ | Panel C: Barplot Vertical)

Esta celda genera una figura compuesta lista para informes y presentaciones (`300 dpi` en formato `.png` y vectorial `.svg`):
* **Panel A (izquierda):** Mapa con límites regionales de Chile (GADM v4.1), EOO/núcleos en verde suave, *inset* de ubicación de Chile continental, barra de escala métrica y **diagramas de torta sin solapamiento** conectados con líneas guía punteadas y el `Codigo` de cada localidad *(más superficie IDW enmascarada si activaste `MOSTRAR_IDW_ANCESTRIA = True`)*.
* **Panel B (arriba derecha):** Curva de **Evanno ($\Delta K$)** junto a $\text{Mean } \ln P(D) \pm \text{SD}$, destacando el $K$ seleccionado.
* **Panel C (abajo derecha):** **Barplot vertical de ancestría individual** ordenado de Norte a Sur (en espejo con la latitud del mapa), etiquetado con los códigos cortos y con la misma paleta de colores del mapa.

> **Referencia del diseño de visualización:** La composición multipanel A + B + C (mapa con diagramas de torta conectados por líneas guía, curva de selección de $K$ y barplot vertical sincronizado) está inspirada en Pereira et al. (2022), *Heredity* 129: 161–168 ([doi:10.1038/s41437-022-00549-7](https://doi.org/10.1038/s41437-022-00549-7)).

In [ ]:
def cargar_contexto_vecino_arg():
    '''Descarga y cachea el contorno simplificado de Argentina (GADM 4.1 nivel 0) como contexto terrestre al Este.'''
    cache_dir = "cache_limites_geo"
    os.makedirs(cache_dir, exist_ok=True)
    zip_arg = os.path.join(cache_dir, "gadm41_ARG_0.json.zip")
    url_arg = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_ARG_0.json.zip"
    try:
        if not os.path.exists(zip_arg):
            r = SESSION_HTTP.get(url_arg, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            with open(zip_arg, "wb") as f:
                f.write(r.content)
        with zipfile.ZipFile(zip_arg, "r") as zf:
            jname = [n for n in zf.namelist() if n.endswith(".json")][0]
            jpath = os.path.join(cache_dir, jname)
            if not os.path.exists(jpath):
                zf.extract(jname, cache_dir)
        gdf_arg = gpd.read_file(jpath).to_crs("EPSG:4326")
        gdf_arg["geometry"] = gdf_arg.geometry.make_valid()
        return gdf_arg
    except Exception:
        return None

def cargar_regiones_chile_gadm():
    '''Descarga y cachea los límites regionales de Chile (GADM 4.1 nivel 1) para el mapa de publicación.'''
    cache_dir = "cache_limites_geo"
    os.makedirs(cache_dir, exist_ok=True)
    zip_reg = os.path.join(cache_dir, "gadm41_CHL_1.json.zip")
    url_reg = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_CHL_1.json.zip"
    try:
        if not os.path.exists(zip_reg):
            r = SESSION_HTTP.get(url_reg, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            with open(zip_reg, "wb") as f:
                f.write(r.content)
        with zipfile.ZipFile(zip_reg, "r") as zf:
            jname = [n for n in zf.namelist() if n.endswith(".json")][0]
            jpath = os.path.join(cache_dir, jname)
            if not os.path.exists(jpath):
                zf.extract(jname, cache_dir)
        gdf_reg = gpd.read_file(jpath).to_crs("EPSG:4326")
        gdf_reg["geometry"] = gdf_reg.geometry.make_valid()
        return gdf_reg
    except Exception as e:
        print(f"Aviso: No se pudo cargar GADM nivel 1 ({e}); se usará la máscara base.")
        return None

def calcular_superficie_idw_rgba(gdf_pop_ea, k_cols, colores_hex, mascara_dom_ea, radio_km=60, potencia=2.0, n_grid=240):
    '''
    Calcula superficie raster RGBA de ancestría interpolada por IDW (tipo tess3r) sobre malla WGS84,
    con distancias exactas en LAEA Equal-Area, doble enmascaramiento (EOO/núcleos + radio muestreado)
    y opacidad modulada por certeza Qmax.
    '''
    mascara_radio = gdf_pop_ea.geometry.buffer(radio_km * 1000.0).union_all()
    mascara_idw_ea = shapely.make_valid(mascara_dom_ea.intersection(mascara_radio))
    if mascara_idw_ea.is_empty:
        return None, None, None

    # Límites en WGS84 para alineación exacta en Folium ImageOverlay y en Matplotlib
    mascara_wgs84 = gpd.GeoSeries([mascara_idw_ea], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]
    min_lon, min_lat, max_lon, max_lat = mascara_wgs84.bounds

    lons = np.linspace(min_lon, max_lon, n_grid)
    lats = np.linspace(min_lat, max_lat, n_grid)
    GLON, GLAT = np.meshgrid(lons, lats)

    # Proyectar grilla a LAEA Equal-Area para máscara y distancias euclidianas en metros
    pts_grid_ea = gpd.GeoSeries(
        gpd.points_from_xy(GLON.ravel(), GLAT.ravel()), crs="EPSG:4326"
    ).to_crs(CRS_EQUAL_AREA)
    gx_ea = pts_grid_ea.x.to_numpy().reshape(GLON.shape)
    gy_ea = pts_grid_ea.y.to_numpy().reshape(GLAT.shape)

    dentro = shapely.contains_xy(mascara_idw_ea, gx_ea, gy_ea)
    rgba = np.zeros((n_grid, n_grid, 4), dtype=np.float32)

    if not np.any(dentro):
        return None, None, None

    px = gdf_pop_ea.geometry.x.to_numpy()
    py = gdf_pop_ea.geometry.y.to_numpy()
    Q = gdf_pop_ea[k_cols].to_numpy(dtype=float)
    K = len(k_cols)
    rgb_pal = np.array([mcolors.to_rgb(c) for c in colores_hex], dtype=np.float32)

    x_in = gx_ea[dentro][:, None]
    y_in = gy_ea[dentro][:, None]
    dist = np.hypot(x_in - px[None, :], y_in - py[None, :])
    w = 1.0 / np.power(dist + 250.0, potencia)
    w = w / np.sum(w, axis=1, keepdims=True)
    Q_interp = w @ Q  # (n_in, K)

    dom = np.argmax(Q_interp, axis=1)
    q_max_in = np.max(Q_interp, axis=1)
    # Certeza normalizada [0, 1] -> opacidad [0.15, 0.62]
    certeza = np.clip((q_max_in - (1.0 / K)) / max(1.0 - (1.0 / K), 1e-6), 0.0, 1.0) if K > 1 else np.ones_like(q_max_in)
    alpha_in = 0.15 + 0.47 * certeza

    rgba_in = np.zeros((len(dom), 4), dtype=np.float32)
    rgba_in[:, :3] = rgb_pal[dom]
    rgba_in[:, 3] = alpha_in
    rgba[dentro] = rgba_in

    # También devolver bounds en EA para graficar limpiamente en Panel A (que está en LAEA)
    bounds_wgs84 = [[min_lat, min_lon], [max_lat, max_lon]]
    return rgba, bounds_wgs84, mascara_idw_ea

def distribuir_posiciones_y(ys_objetivo, y_min, y_max, sep_min):
    '''Ajusta coordenadas Y ordenadas (Norte->Sur) garantizando separación mínima sin solapamiento.'''
    n = len(ys_objetivo)
    if n == 0:
        return []
    if n == 1:
        return [float(np.clip(ys_objetivo[0], y_min, y_max))]

    # Si no cabe con el espaciado ideal, distribuir uniformemente en el rango disponible
    if (n - 1) * sep_min >= (y_max - y_min):
        return list(np.linspace(y_max, y_min, n))

    # Relajación iterativa preservando orden Norte->Sur y cercanía a la latitud real
    ys = np.clip(np.array(ys_objetivo, dtype=float), y_min, y_max)
    for _ in range(120):
        # Empujar hacia abajo si están muy cerca (orden decreciente: ys[i] >= ys[i+1] + sep_min)
        for i in range(n - 1):
            gap = ys[i] - ys[i + 1]
            if gap < sep_min:
                dif = (sep_min - gap) / 2.0
                ys[i] += dif
                ys[i + 1] -= dif
        # Re-clamping a los límites
        if ys[0] > y_max:
            ys -= (ys[0] - y_max)
        if ys[-1] < y_min:
            ys += (y_min - ys[-1])
        ys = np.clip(ys, y_min, y_max)
    return ys.tolist()

if not globals().get("STRUCTURE_LISTO", False):
    print("Aún no se ha ejecutado la celda 11.1 (o el entorno de Colab se reinició). Ejecuta las celdas previas y la 11.1 con ACTIVAR_MODULO_STRUCTURE = True.")
else:
    # 1. Calcular IDW si está activado y aún no fue precalculado en 11.1
    if MOSTRAR_IDW_ANCESTRIA and globals().get("IDW_RGBA_WGS84") is None:
        if es_disyunta and gdf_eoo_subpob_ea is not None:
            dom_idw = gdf_eoo_subpob_ea.geometry.union_all().union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        else:
            dom_idw = eoo_mcp_tierra_geom.union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        IDW_RGBA_WGS84, IDW_BOUNDS_WGS84, _ = calcular_superficie_idw_rgba(
            gdf_struct_pop_ea, K_COLS_ACTIVOS, COLORES_ACTIVOS,
            mascara_dom_ea=dom_idw, radio_km=RADIO_MASCARA_IDW_KM, potencia=POTENCIA_IDW
        )

    # 2. Cargar regiones de Chile (GADM 4.1 nivel 1) + contexto terrestre de Argentina
    gdf_regiones_wgs84 = cargar_regiones_chile_gadm()
    gdf_arg_wgs84 = cargar_contexto_vecino_arg()

    # Definir caja espacial en LAEA Equal-Area con pasillos laterales amplios para las tortas
    pts_union_bounds = gdf_struct_pop_ea.total_bounds
    eoo_bounds = gdf_eoo_oficial_ea.total_bounds
    x_min_d = min(pts_union_bounds[0], eoo_bounds[0])
    y_min_d = min(pts_union_bounds[1], eoo_bounds[1])
    x_max_d = max(pts_union_bounds[2], eoo_bounds[2])
    y_max_d = max(pts_union_bounds[3], eoo_bounds[3])

    span_x = max(x_max_d - x_min_d, 120000.0)
    span_y = max(y_max_d - y_min_d, 180000.0)

    # Corredores izquierdo y derecho para ubicar las tortas sin tapar el territorio ni el inset
    margen_y_top = 0.14 * span_y
    margen_y_bot = 0.16 * span_y
    ylim_bot = y_min_d - margen_y_bot
    ylim_top = y_max_d + margen_y_top
    altura_total = ylim_top - ylim_bot

    # Radio de torta equilibrado para garantizar holgura vertical entre todas las tortas
    n_pops = len(gdf_struct_pop_ea)
    n_lado = math.ceil(n_pops / 2)
    r_pie = min(altura_total / (4.15 * max(n_lado, 4)), 0.048 * altura_total)

    x_corredor_izq = x_min_d - max(0.32 * span_x, 2.5 * r_pie)
    x_corredor_der = x_max_d + max(0.32 * span_x, 2.5 * r_pie)
    xlim_izq = x_corredor_izq - 2.60 * r_pie
    xlim_der = x_corredor_der + 2.55 * r_pie

    # 3. Configurar figura compuesta (Panel A izquierda | Panel B arriba-der | Panel C abajo-der)
    fig = plt.figure(figsize=(14.8, 10.8), dpi=150, facecolor="white")
    gs = gridspec.GridSpec(
        2, 2,
        width_ratios=[2.25, 1.0],
        height_ratios=[0.27, 0.73],
        wspace=0.14, hspace=0.24
    )

    ax_map = fig.add_subplot(gs[:, 0])
    ax_ev  = fig.add_subplot(gs[0, 1])
    ax_bar = fig.add_subplot(gs[1, 1])

    # ================= PANEL A: MAPA + TORTAS SIN SOLAPAMIENTO =================
    ax_map.set_facecolor("#E5F0F9")  # Océano Pacífico suave
    bbox_plot_ea = box(xlim_izq - 90000, ylim_bot - 90000, xlim_der + 90000, ylim_top + 90000)
    bbox_plot_wgs84 = gpd.GeoSeries([bbox_plot_ea], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]

    # Contexto continental terrestre (Argentina en gris cálido muy tenue para no confundirlo con océano)
    if gdf_arg_wgs84 is not None:
        arg_clip = gpd.clip(gdf_arg_wgs84, bbox_plot_wgs84)
        if not arg_clip.empty:
            arg_clip.to_crs(CRS_EQUAL_AREA).plot(
                ax=ax_map, facecolor="#ECECE8", edgecolor="#B5B5B0", linewidth=0.6, zorder=1
            )

    # Regiones de Chile (GADM 4.1 nivel 1)
    if gdf_regiones_wgs84 is not None:
        reg_clip = gpd.clip(gdf_regiones_wgs84, bbox_plot_wgs84).to_crs(CRS_EQUAL_AREA)
        reg_clip.plot(ax=ax_map, facecolor="#FAF9F5", edgecolor="#888888", linewidth=0.65, zorder=2)
        # Etiquetas discretas de regiones
        for _, reg_row in reg_clip.iterrows():
            if reg_row.geometry.area / 1e6 > 1500:
                pt_r = reg_row.geometry.representative_point()
                if (x_min_d - 0.15 * span_x <= pt_r.x <= x_max_d + 0.15 * span_x) and (y_min_d <= pt_r.y <= y_max_d):
                    nom_reg = str(reg_row.get("NAME_1", "")).replace("Región de ", "").replace("Región del ", "")
                    ax_map.text(
                        pt_r.x, pt_r.y, nom_reg, fontsize=6.5, color="#777777",
                        ha="center", va="center", style="italic", zorder=3,
                        path_effects=[withStroke(linewidth=1.8, foreground="#FAF9F5")]
                    )
    else:
        gpd.GeoSeries([mascara_tierra_ea], crs=CRS_EQUAL_AREA).plot(
            ax=ax_map, facecolor="#FAF9F5", edgecolor="#666666", linewidth=0.8, zorder=2
        )

    # EOO / núcleos biogeográficos en verde salvia suave (referencia visual del rango)
    capa_rango = gdf_eoo_subpob_ea if (es_disyunta and gdf_eoo_subpob_ea is not None) else gdf_eoo_oficial_ea
    capa_rango.plot(ax=ax_map, facecolor="#C7E2C2", edgecolor="#4F8A4B", linewidth=1.1, alpha=0.58, zorder=4)

    # Superficie IDW opcional (si el usuario la activó)
    if MOSTRAR_IDW_ANCESTRIA and IDW_RGBA_WGS84 is not None:
        mascara_radio_ea = gdf_struct_pop_ea.geometry.buffer(RADIO_MASCARA_IDW_KM * 1000.0).union_all()
        dom_ea = (gdf_eoo_subpob_ea.geometry.union_all() if (es_disyunta and gdf_eoo_subpob_ea is not None) else eoo_mcp_tierra_geom)
        dom_ea = dom_ea.union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        m_idw = shapely.make_valid(dom_ea.intersection(mascara_radio_ea))
        if not m_idw.is_empty:
            mx0, my0, mx1, my1 = m_idw.bounds
            gx1 = np.linspace(mx0, mx1, 220)
            gy1 = np.linspace(my0, my1, 220)
            GXX, GYY = np.meshgrid(gx1, gy1)
            mask_in = shapely.contains_xy(m_idw, GXX, GYY)
            rgba_ea = np.zeros((220, 220, 4), dtype=np.float32)
            if np.any(mask_in):
                px = gdf_struct_pop_ea.geometry.x.to_numpy()
                py = gdf_struct_pop_ea.geometry.y.to_numpy()
                Q_m = gdf_struct_pop_ea[K_COLS_ACTIVOS].to_numpy(dtype=float)
                rgb_p = np.array([mcolors.to_rgb(c) for c in COLORES_ACTIVOS], dtype=np.float32)
                d_ea = np.hypot(GXX[mask_in][:, None] - px[None, :], GYY[mask_in][:, None] - py[None, :])
                w_ea = 1.0 / np.power(d_ea + 250.0, POTENCIA_IDW)
                w_ea /= np.sum(w_ea, axis=1, keepdims=True)
                Qi = w_ea @ Q_m
                dom_i = np.argmax(Qi, axis=1)
                qm_i = np.max(Qi, axis=1)
                cert_i = np.clip((qm_i - 1.0 / K_ACTIVO) / max(1.0 - 1.0 / K_ACTIVO, 1e-6), 0.0, 1.0) if K_ACTIVO > 1 else np.ones_like(qm_i)
                rgba_ea[mask_in, :3] = rgb_p[dom_i]
                rgba_ea[mask_in, 3] = 0.14 + 0.44 * cert_i
                ax_map.imshow(rgba_ea, origin="lower", extent=[mx0, mx1, my0, my1], zorder=5)

    # Asignación balanceada Izquierda / Derecha ordenada de Norte a Sur (0 cruces de líneas guía)
    df_p_ord = gdf_struct_pop_ea.sort_values("Lat", ascending=False).reset_index(drop=True)
    izq_rows, der_rows = [], []
    i = 0
    while i < len(df_p_ord):
        if i + 1 < len(df_p_ord):
            r1, r2 = df_p_ord.iloc[i], df_p_ord.iloc[i + 1]
            if r1.geometry.x <= r2.geometry.x:
                izq_rows.append(r1)
                der_rows.append(r2)
            else:
                izq_rows.append(r2)
                der_rows.append(r1)
            i += 2
        else:
            r1 = df_p_ord.iloc[i]
            if len(izq_rows) <= len(der_rows):
                izq_rows.append(r1)
            else:
                der_rows.append(r1)
            i += 1

    izq_rows = sorted(izq_rows, key=lambda r: r.geometry.y, reverse=True)
    der_rows = sorted(der_rows, key=lambda r: r.geometry.y, reverse=True)

    # Reservar espacio despejado para inset (superior-izq) y barra de escala (inferior-izq)
    y_izq_max = ylim_top - 0.34 * altura_total
    y_izq_min = ylim_bot + 0.13 * altura_total
    y_der_max = ylim_top - 0.08 * altura_total
    y_der_min = ylim_bot + 0.08 * altura_total

    ys_izq = distribuir_posiciones_y([r.geometry.y for r in izq_rows], y_izq_min, y_izq_max, sep_min=2.65 * r_pie)
    ys_der = distribuir_posiciones_y([r.geometry.y for r in der_rows], y_der_min, y_der_max, sep_min=2.65 * r_pie)

    def dibujar_torta_con_guia(ax, row, cx, cy, radio, lado):
        px, py = row.geometry.x, row.geometry.y
        ax.scatter([px], [py], color="black", s=24, zorder=8, edgecolors="white", linewidths=0.6)
        ax.plot([px, cx], [py, cy], color="#2F2F2F", linestyle=":", linewidth=1.05, zorder=6)
        ang_ini = 90.0
        for idx_k, col_k in enumerate(K_COLS_ACTIVOS):
            frac = float(row[col_k])
            if frac <= 0:
                continue
            ang_fin = ang_ini - frac * 360.0
            wedge = mpatches.Wedge(
                (cx, cy), radio, ang_fin, ang_ini,
                facecolor=COLORES_ACTIVOS[idx_k], edgecolor="#111111", linewidth=0.75, zorder=9
            )
            ax.add_patch(wedge)
            ang_ini = ang_fin
        circ = mpatches.Circle((cx, cy), radio, facecolor="none", edgecolor="#111111", linewidth=0.9, zorder=10)
        ax.add_patch(circ)
        if lado == "izq":
            tx, ha = cx - 1.22 * radio, "right"
        else:
            tx, ha = cx + 1.22 * radio, "left"
        ax.text(
            tx, cy, str(row["Codigo"]), fontsize=8.5, fontweight="bold", color="#111111",
            ha=ha, va="center", zorder=11,
            path_effects=[withStroke(linewidth=2.4, foreground="white")]
        )

    for row, cy in zip(izq_rows, ys_izq):
        dibujar_torta_con_guia(ax_map, row, x_corredor_izq, cy, r_pie, lado="izq")
    for row, cy in zip(der_rows, ys_der):
        dibujar_torta_con_guia(ax_map, row, x_corredor_der, cy, r_pie, lado="der")

    ax_map.set_xlim(xlim_izq, xlim_der)
    ax_map.set_ylim(ylim_bot, ylim_top)
    ax_map.set_aspect("equal")
    ax_map.set_xticks([])
    ax_map.set_yticks([])
    ax_map.set_xlabel("")
    ax_map.set_ylabel("")
    for spine in ax_map.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.0)

    titulo_idw = " + IDW exploratorio" if MOSTRAR_IDW_ANCESTRIA else ""
    ax_map.set_title(
        f"A   Estructura genética ({ESPECIE_SELECCIONADA}, K = {K_ACTIVO}){titulo_idw}",
        loc="left", fontsize=11.5, fontweight="bold", pad=8
    )

    # Barra de escala métrica exacta en km (esquina inferior izquierda)
    ancho_mapa_km = (xlim_der - xlim_izq) / 1000.0
    largo_esc_km = next((v for v in [20, 50, 100, 150, 200, 300] if v >= ancho_mapa_km * 0.16), 100)
    esc_x0 = xlim_izq + 0.04 * (xlim_der - xlim_izq)
    esc_y0 = ylim_bot + 0.038 * altura_total
    esc_h = 0.011 * altura_total
    half_m = (largo_esc_km * 1000.0) / 2.0
    ax_map.add_patch(mpatches.Rectangle((esc_x0, esc_y0), half_m, esc_h, facecolor="black", edgecolor="black", zorder=12))
    ax_map.add_patch(mpatches.Rectangle((esc_x0 + half_m, esc_y0), half_m, esc_h, facecolor="white", edgecolor="black", zorder=12))
    for val_km, pos_x in [(0, esc_x0), (largo_esc_km // 2, esc_x0 + half_m), (largo_esc_km, esc_x0 + 2 * half_m)]:
        lbl = f"{val_km} km" if val_km == largo_esc_km else f"{val_km}"
        ax_map.text(pos_x, esc_y0 + esc_h * 1.45, lbl, fontsize=7.5, ha="center", va="bottom", fontweight="bold", zorder=12)

    # Leyenda horizontal limpia DEBAJO del Panel A (cero solapamiento con cualquier torta o K)
    handles_leg = [
        mpatches.Patch(facecolor="#C7E2C2", edgecolor="#4F8A4B", label="Rango EOO / Núcleos")
    ]
    for idx_k in range(K_ACTIVO):
        suf_pos = " (Norte)" if idx_k == 0 and K_ACTIVO > 1 else (" (Sur)" if idx_k == K_ACTIVO - 1 and K_ACTIVO > 1 else "")
        handles_leg.append(
            mpatches.Patch(facecolor=COLORES_ACTIVOS[idx_k], edgecolor="#111111", label=f"Cluster K{idx_k+1}{suf_pos}")
        )
    ax_map.legend(
        handles=handles_leg, loc="upper center", bbox_to_anchor=(0.5, -0.018),
        ncol=min(len(handles_leg), 4), frameon=True, facecolor="white",
        edgecolor="#666666", fontsize=8.2, handlelength=1.4, columnspacing=1.2
    )

    # Inset de ubicación de Chile continental (esquina superior izquierda, sin etiquetas de ejes)
    if gdf_regiones_wgs84 is not None:
        ax_ins = ax_map.inset_axes([0.02, 0.72, 0.16, 0.25])
        ax_ins.set_facecolor("#E5F0F9")
        chile_simple = gdf_regiones_wgs84.dissolve().simplify(0.08)
        chile_simple.plot(ax=ax_ins, facecolor="#D6D6D2", edgecolor="#555555", linewidth=0.5)
        b_est = gdf_struct_pop_wgs84.total_bounds
        rec = mpatches.Rectangle(
            (b_est[0] - 0.8, b_est[1] - 0.8),
            (b_est[2] - b_est[0]) + 1.6,
            (b_est[3] - b_est[1]) + 1.6,
            facecolor="none", edgecolor="#C0392B", linewidth=1.6, zorder=5
        )
        ax_ins.add_patch(rec)
        ax_ins.set_xlim(-76.5, -66.0)
        ax_ins.set_ylim(-56.5, -17.0)
        ax_ins.set_xticks([])
        ax_ins.set_yticks([])
        ax_ins.set_xlabel("")
        ax_ins.set_ylabel("")
        ax_ins.set_title("Chile", fontsize=7.5, fontweight="bold", pad=2)
        for sp in ax_ins.spines.values():
            sp.set_edgecolor("#333333")
            sp.set_linewidth(0.8)

    # ================= PANEL B: EVANNO (DELTA K + LN P(D)) =================
    if df_evanno is not None and len(df_evanno) >= 2:
        df_dk = df_evanno.dropna(subset=["delta_K"])
        if not df_dk.empty:
            l1, = ax_ev.plot(df_dk["K"], df_dk["delta_K"], color="#111111", marker="o", linewidth=1.8, markersize=5, label="ΔK (Evanno)")
            l2 = ax_ev.axvline(K_ACTIVO, color="#C0392B", linestyle="--", linewidth=1.3, label=f"K elegido = {K_ACTIVO}")
            ax_ev.set_ylabel("ΔK (Evanno)", fontsize=8.5, fontweight="bold", color="#111111")
            ax_ev.set_xticks(df_evanno["K"].astype(int).tolist())
            max_dk_val = float(df_dk["delta_K"].max())
            ax_ev.set_ylim(0, max(max_dk_val * 1.28, 1.0))

            ax_ev2 = ax_ev.twinx()
            l3 = ax_ev2.errorbar(
                df_evanno["K"], df_evanno["mean_lnpd"], yerr=df_evanno["sd_lnpd"].fillna(0),
                color="#7F8C8D", linestyle=":", marker="s", markersize=3.5, linewidth=1.0, capsize=2, alpha=0.85,
                label="Mean Ln P(D)"
            )
            ax_ev2.set_ylabel("Mean Ln P(D)", fontsize=7.5, color="#7F8C8D")
            ax_ev2.tick_params(axis="y", labelcolor="#7F8C8D", labelsize=7)
            ax_ev2.spines["top"].set_visible(False)
            ln_min = float((df_evanno["mean_lnpd"] - df_evanno["sd_lnpd"].fillna(0)).min())
            ln_max = float((df_evanno["mean_lnpd"] + df_evanno["sd_lnpd"].fillna(0)).max())
            ln_pad = max((ln_max - ln_min) * 0.25, 10.0)
            ax_ev2.set_ylim(ln_min - ln_pad * 0.4, ln_max + ln_pad)

            ax_ev.legend(
                handles=[l1, l2, l3], labels=["ΔK (Evanno)", f"K = {K_ACTIVO}", "Mean Ln P(D)"],
                loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=3,
                fontsize=6.8, frameon=False, handlelength=1.4, columnspacing=0.8
            )
        else:
            ax_ev.errorbar(
                df_evanno["K"], df_evanno["mean_lnpd"], yerr=df_evanno["sd_lnpd"].fillna(0),
                color="#111111", marker="o", linewidth=1.6, capsize=3
            )
            ax_ev.axvline(K_ACTIVO, color="#C0392B", linestyle="--", linewidth=1.3)
            ax_ev.set_ylabel("Mean Ln P(D)", fontsize=8.5, fontweight="bold")

        ax_ev.set_xlabel("Número de grupos genéticos (K)", fontsize=8.5)
        ax_ev.tick_params(labelsize=7.5)
        ax_ev.spines["top"].set_visible(False)
        ax_ev.set_title("B   Selección de K (Evanno / Ln P(D))", loc="left", fontsize=10, fontweight="bold", pad=12)
    else:
        ax_ev.axis("off")
        ax_ev.text(
            0.5, 0.5, f"Corrida única / CSV\nK seleccionado = {K_ACTIVO}\n(Sube el .zip de Results/\npara calcular Evanno ΔK)",
            ha="center", va="center", fontsize=9, color="#444444",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#F4F6F6", edgecolor="#BBBBBB")
        )
        ax_ev.set_title("B   Resumen de K", loc="left", fontsize=10, fontweight="bold", pad=6)

    # ================= PANEL C: BARPLOT VERTICAL SINCRONIZADO =================
    if ORDEN_BARPLOT == "NORTE_A_SUR":
        orden_pops = df_pop_join.sort_values("Lat", ascending=False)["Pop"].tolist()
    else:
        orden_pops = df_pop_join.sort_values("Pop", ascending=True)["Pop"].tolist()

    bloques = []
    etiquetas_y = []
    separadores = []
    pos_act = 0
    for pid in orden_pops:
        sub_ind = df_struct_ind[df_struct_ind["Pop"] == pid].copy()
        bloques.append(sub_ind)
        n_b = len(sub_ind)
        cod = str(df_pop_join.loc[df_pop_join["Pop"] == pid, "Codigo"].iloc[0])
        etiquetas_y.append((pos_act + n_b / 2.0, cod))
        pos_act += n_b
        separadores.append(pos_act)

    df_bar = pd.concat(bloques, ignore_index=True)
    y_idx = np.arange(len(df_bar))
    left_acum = np.zeros(len(df_bar), dtype=float)

    for idx_k, col_k in enumerate(K_COLS_ACTIVOS):
        vals = df_bar[col_k].to_numpy(dtype=float)
        ax_bar.barh(
            y_idx, vals, left=left_acum, height=1.0,
            color=COLORES_ACTIVOS[idx_k], edgecolor="none"
        )
        left_acum += vals

    for sep_y in separadores[:-1]:
        ax_bar.axhline(sep_y - 0.5, color="white", linewidth=1.1)

    ax_bar.set_ylim(len(df_bar) - 0.5, -0.5)  # Norte arriba, Sur abajo
    ax_bar.set_xlim(0.0, 1.0)
    ax_bar.set_yticks([p for p, _ in etiquetas_y])
    ax_bar.set_yticklabels([c for _, c in etiquetas_y], fontsize=8.5, fontweight="bold")
    ax_bar.set_xlabel("Proporción de ancestría (Q)", fontsize=8.5)
    ax_bar.set_xticks([0.0, 0.5, 1.0])
    ax_bar.tick_params(axis="x", labelsize=7.5)
    for sp in ax_bar.spines.values():
        sp.set_edgecolor("#222222")
        sp.set_linewidth(0.8)
    ax_bar.set_title(f"C   Ancestría individual ({'Norte → Sur' if ORDEN_BARPLOT == 'NORTE_A_SUR' else 'Orden ID'})", loc="left", fontsize=10, fontweight="bold", pad=6)

    fig_png = f"figura_structure_{ESPECIE_SELECCIONADA.lower()}_K{K_ACTIVO}.png"
    fig_svg = f"figura_structure_{ESPECIE_SELECCIONADA.lower()}_K{K_ACTIVO}.svg"
    fig.savefig(fig_png, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(fig_svg, bbox_inches="tight", facecolor="white")
    print(f"Figura compuesta guardada en alta resolución: '{fig_png}' (300 dpi) y '{fig_svg}' (vectorial).")
    plt.show()

### 11.3. Mapa Interactivo Integrado (`Folium`): Distribución, Biodiversidad y Tortas SVG de STRUCTURE

Integra en un único visor interactivo todas las capas anteriores junto con:
* **Diagramas de torta SVG vectoriales** por localidad (haciendo clic se despliega el detalle de porcentajes $K_1 \dots K_K$, $Q_{\max}$, entropía $H_Q$ y estado *Pura* vs. *Mezcla*).
* **Superficie IDW de ancestría** *(solo si activaste `MOSTRAR_IDW_ANCESTRIA = True`, recortada a los núcleos/EOO y al radio muestreado)*.

In [ ]:
def construir_svg_torta(valores_q, colores, tamano=42):
    '''Genera un SVG limpio de gráfico de torta para incrustar como DivIcon en Folium.'''
    r = tamano / 2.0 - 2.0
    cx = cy = tamano / 2.0
    total = sum(valores_q)
    if total <= 0:
        return f'<svg width="{tamano}" height="{tamano}"><circle cx="{cx}" cy="{cy}" r="{r}" fill="#999"/></svg>'

    partes = [f'<svg width="{tamano}" height="{tamano}" viewBox="0 0 {tamano} {tamano}" xmlns="http://www.w3.org/2000/svg">']
    # Si un único cluster domina ~100%, dibujar círculo sólido limpio
    if max(valores_q) / total >= 0.999:
        idx_dom = int(np.argmax(valores_q))
        partes.append(f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="{colores[idx_dom]}" stroke="#111" stroke-width="1.4"/>')
    else:
        ang = -math.pi / 2.0
        for val, col in zip(valores_q, colores):
            frac = val / total
            if frac <= 1e-4:
                continue
            ang_sig = ang + 2.0 * math.pi * frac
            x1, y1 = cx + r * math.cos(ang), cy + r * math.sin(ang)
            x2, y2 = cx + r * math.cos(ang_sig), cy + r * math.sin(ang_sig)
            large_arc = 1 if frac > 0.5 else 0
            d = f"M {cx:.2f} {cy:.2f} L {x1:.2f} {y1:.2f} A {r:.2f} {r:.2f} 0 {large_arc} 1 {x2:.2f} {y2:.2f} Z"
            partes.append(f'<path d="{d}" fill="{col}" stroke="#111" stroke-width="0.9"/>')
            ang = ang_sig
        partes.append(f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="#111" stroke-width="1.3"/>')
    partes.append("</svg>")
    return "".join(partes)

if not globals().get("STRUCTURE_LISTO", False):
    print("Aún no se ha ejecutado la celda 11.1 (o el entorno de Colab se reinició). Ejecuta 'Entorno de ejecución -> Ejecutar todo' o corre desde la celda 11.1 con ACTIVAR_MODULO_STRUCTURE = True.")
else:
    m_gen = folium.Map(location=[lat_centro, lon_centro], zoom_start=8, tiles=None)

    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        attr="&copy; OpenStreetMap contributors &copy; CARTO",
        name="Mapa Claro (CartoDB Positron)",
        show=True,
    ).add_to(m_gen)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",
        name="Topográfico (Esri World Topo)",
        show=False,
    ).add_to(m_gen)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",
        name="Satélite (Esri World Imagery)",
        show=False,
    ).add_to(m_gen)

    # Capas base de conservación (apagadas las más densas para dar protagonismo a STRUCTURE)
    gdf_hex_final_ea.to_crs("EPSG:4326").explore(
        m=m_gen,
        name="Biodiversidad Flora Vascular (Shannon H')",
        column="shannon_h",
        cmap="viridis",
        tooltip=["hex_id", "n_registros", "riqueza_s", "shannon_h", "pielou_j", "simpson_1_d", "cobertura_c", "sp_dominante"],
        legend=False,
        style_kwds={"fillOpacity": 0.45, "weight": 0.7},
        show=False,
    )
    gdf_eoo_oficial_ea.to_crs("EPSG:4326").explore(
        m=m_gen,
        name=f"EOO Oficial B1 ({eoo_oficial_km2:,.0f} km²)",
        color="darkorange",
        style_kwds={"fill": True, "fillOpacity": 0.10, "weight": 2.2},
        tooltip=["especie", "escala", "eoo_km2", "umbral_preliminar_b1"],
        show=True,
    )
    if gdf_eoo_subpob_ea is not None:
        gdf_eoo_subpob_ea.to_crs("EPSG:4326").explore(
            m=m_gen,
            name="Núcleos Biogeográficos",
            color="royalblue",
            style_kwds={"fill": True, "fillOpacity": 0.10, "weight": 1.8, "dashArray": "5, 5"},
            tooltip=["subpoblacion", "n_registros", "area_nucleo_km2"],
            show=False,
        )
    gdf_aoo_uicn_ea.to_crs("EPSG:4326").explore(
        m=m_gen,
        name=f"AOO Oficial 2×2 km ({stats_aoo_uicn['aoo_km2']:,.0f} km²)",
        color="crimson",
        style_kwds={"fillOpacity": 0.35, "weight": 0.8},
        tooltip=["celda_id", "aoo_total_km2", "umbral_preliminar_b2"],
        show=False,
    )

    # Capa IDW exploratoria opcional (ImageOverlay con inversión vertical para origen norte de PNG)
    _idw_rgba = globals().get("IDW_RGBA_WGS84")
    _idw_bounds = globals().get("IDW_BOUNDS_WGS84")
    if MOSTRAR_IDW_ANCESTRIA and _idw_rgba is not None and _idw_bounds is not None:
        buf_png = io.BytesIO()
        plt.imsave(buf_png, np.flipud(_idw_rgba), format="png")
        b64_png = base64.b64encode(buf_png.getvalue()).decode("ascii")
        folium.raster_layers.ImageOverlay(
            image=f"data:image/png;base64,{b64_png}",
            bounds=_idw_bounds,
            opacity=0.9,
            name=f"Ancestría IDW Exploratoria (radio {RADIO_MASCARA_IDW_KM} km)",
            interactive=False,
            show=True,
        ).add_to(m_gen)

    # Capa de tortas SVG vectoriales de STRUCTURE por localidad
    fg_pies = folium.FeatureGroup(name=f"STRUCTURE Tortas por Localidad (K = {K_ACTIVO})", show=True)
    for _, row in gdf_struct_pop_wgs84.iterrows():
        q_vals = [float(row[c]) for c in K_COLS_ACTIVOS]
        svg_html = construir_svg_torta(q_vals, COLORES_ACTIVOS, tamano=40)
        icon_html = (
            f'<div style="display:flex;flex-direction:column;align-items:center;">'
            f'{svg_html}'
            f'<span style="font-size:10px;font-weight:700;color:#111;background:rgba(255,255,255,0.85);'
            f'padding:0px 3px;border-radius:3px;border:1px solid #999;margin-top:-3px;">{row["Codigo"]}</span>'
            f'</div>'
        )
        filas_q = "".join(
            f'<tr><td><span style="display:inline-block;width:10px;height:10px;background:{COLORES_ACTIVOS[i]};'
            f'border:1px solid #222;margin-right:5px;"></span><b>K{i+1}</b></td>'
            f'<td style="text-align:right;padding-left:10px;">{q_vals[i]*100:.1f}%</td></tr>'
            for i in range(K_ACTIVO)
        )
        popup_html = (
            f'<div style="font-family:sans-serif;font-size:12px;min-width:190px;">'
            f'<b>{row["Codigo"]} — {row["Referencia"]}</b><br>'
            f'Pop #{int(row["Pop"])} | n = {int(row["n_ind"])} ind.<br>'
            f'<table style="margin-top:5px;width:100%;">{filas_q}</table>'
            f'<hr style="margin:5px 0;">'
            f'<b>Dominante:</b> {row["cluster_dom"]} (Q<sub>max</sub>={row["q_max"]:.2f})<br>'
            f'<b>Estado:</b> {row["estado_mezcla"]}<br>'
            f'<b>Entropía mezcla (H<sub>Q</sub>):</b> {row["entropia_q"]:.3f}'
            f'</div>'
        )
        folium.Marker(
            location=[float(row["Lat"]), float(row["Lon"])],
            icon=folium.DivIcon(html=icon_html, icon_size=(46, 54), icon_anchor=(23, 20)),
            popup=folium.Popup(popup_html, max_width=260),
            tooltip=f"{row['Codigo']} ({row['Referencia']}) — {row['cluster_dom']} ({row['q_max']*100:.0f}%)",
        ).add_to(fg_pies)
    fg_pies.add_to(m_gen)

    # Leyenda flotante compacta de clusters genéticos
    items_ley = "".join(
        f'<div style="margin:2px 0;"><span style="display:inline-block;width:12px;height:12px;'
        f'background:{COLORES_ACTIVOS[i]};border:1px solid #222;vertical-align:middle;margin-right:6px;"></span>'
        f'Cluster K{i+1}</div>'
        for i in range(K_ACTIVO)
    )
    leyenda_html = (
        f'<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:rgba(255,255,255,0.94);'
        f'padding:8px 12px;border:1px solid #555;border-radius:5px;font-family:sans-serif;font-size:12px;'
        f'box-shadow:0 2px 6px rgba(0,0,0,0.2);">'
        f'<b>STRUCTURE ({ESPECIE_SELECCIONADA}, K={K_ACTIVO})</b>{items_ley}</div>'
    )
    m_gen.get_root().html.add_child(folium.Element(leyenda_html))
    folium.LayerControl(collapsed=False).add_to(m_gen)

    mapa_gen_html = f"mapa_interactivo_structure_{ESPECIE_SELECCIONADA.lower()}_K{K_ACTIVO}.html"
    m_gen.save(mapa_gen_html)
    print(f"Mapa interactivo integrado guardado como archivo independiente: '{mapa_gen_html}'")
    display(m_gen)

---

## Referencias Bibliográficas Principales

* **Chao, A., & Jost, L. (2012).** Coverage-based rarefaction and extrapolation: standardizing samples by completeness rather than size. *Ecology*, 93(12), 2533–2547. https://doi.org/10.1890/11-1952.1
* **Evanno, G., Regnaut, S., & Goudet, J. (2005).** Detecting the number of clusters of individuals using the software STRUCTURE: a simulation study. *Molecular Ecology*, 14(8), 2611–2620. https://doi.org/10.1111/j.1365-294X.2005.02553.x
* **IUCN Standards and Petitions Committee (2024).** *Guidelines for Using the IUCN Red List Categories and Criteria*. Version 16. https://www.iucnredlist.org/documents/RedListGuidelines.pdf
* **IUCN (2012).** *Guidelines for Application of IUCN Red List Criteria at Regional and National Levels*. Version 4.0. Gland, Switzerland and Cambridge, UK: IUCN.
* **MMA — Ministerio del Medio Ambiente de Chile (2011).** Decreto Supremo N° 29/2011: *Reglamento para la Clasificación de Especies Silvestres según Estado de Conservación (RCE)*.
* **MMA — Ministerio del Medio Ambiente de Chile (2020).** Decreto Supremo N° 42/2020: *Plan de Recuperación, Conservación y Gestión del Ruil (Nothofagus alessandrii)*.
* **MMA – FAO – GEF (2022).** *Plan de Recuperación, Conservación y Gestión del Queule (Gomortega keule)*. Proyecto Iniciativa Conservación de Especies Amenazadas, Chile (pp. 5–7; LEP-UdeC, 2019).
* **Pereira, A. G., Ferreira, M. F. S., da Silveira, T. C., Soler-Guilhen, J. H., Canal, G. B., Alves, L. B., de Almeida, F. A. N., Gaiotto, F. A., & Ferreira, A. (2022).** Patterns of genetic diversity and structure of a threatened palm species (*Euterpe edulis* Arecaceae) from the Brazilian Atlantic Forest. *Heredity*, 129, 161–168. https://doi.org/10.1038/s41437-022-00549-7 *(referencia del diseño de figura compuesta A + B + C)*.
* **Pritchard, J. K., Stephens, M., & Donnelly, P. (2000).** Inference of population structure using multilocus genotype data. *Genetics*, 155(2), 945–959. https://doi.org/10.1093/genetics/155.2.945